In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/latest.pt
/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/log.jsonl
/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/__huggingface_repos__.json
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data/validation_windows.pt
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data/split_manifest.json
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data/train_windows.pt
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_baseline/best_strict_macro_f1.pt
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_baseline/validation_error_analysis.json
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/.cache/huggingface/.gitignore
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/.cache/huggingface/CACHEDIR.TAG
/kaggle/input/

In [2]:
#!/usr/bin/env python3
"""
phase2_backbones.py -- load the Phase-1 Arch 1 / 2 / 3 backbones for Phase-2 NER and test them.

The three model definitions below are copied VERBATIM from your Phase-1 training scripts
(Arch 1: train_model_1_initfix.py, Arch 2: your Pre-LN script, Arch 3: train_model_3_swa_v2.py).
Each is executed in its own namespace because they reuse class names.

What this file adds (no modelling decisions, no head, no adapters):
  * load_backbone(name, ckpt_path)  -> Backbone(model, hidden(ids), max_len, d_model, meta)
        hidden(ids) returns the (B, T, d_model) states your NER head would read:
          arch1 / arch3: after final_norm, exactly what lm_head reads
          arch2        : model.encode(ids) = encoder output, RAW (this model has no final norm)
  * selftest(name, ckpt_path)       -> strict load, parameter count, checkpoint step/best,
        lm_head(hidden(x)) == model(x) parity (arch1/3), causality (arch1/3),
        bidirectionality (arch2), and activation-scale statistics.

Kaggle: paste into one cell and run; paths default to the ones you sent and can be overridden with
env vars CKPT_ARCH1 / CKPT_ARCH2 / CKPT_ARCH3. Or save as phase2_backbones.py and `import` it.
"""
import math
import os
from types import SimpleNamespace

import torch
import torch.nn as nn
import torch.nn.functional as F

VOCAB = 40000
EXPECTED_PARAMS = {"arch1": 88_210_944, "arch2": 129_957_888, "arch3": 242_501_632}

DEFAULT_CKPTS = {
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
}

# ======================= verbatim model definitions (one namespace each) =======================
_ARCH1_SRC = r'''INIT_STD = 0.02


def _init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0.0, std=INIT_STD)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Embedding):
        nn.init.normal_(m.weight, mean=0.0, std=INIT_STD)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(y)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, block_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = FeedForward(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class LegalDecoderLM(nn.Module):
    def __init__(self, vocab_size, block_size, d_model, n_layers, n_heads, d_ff):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.d_model = d_model

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, block_size)
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight      # tied embeddings

        self.apply(_init_weights)                        # CHANGE 1: small-std init

    def forward(self, x):
        B, T = x.shape
        assert T <= self.block_size

        tok = self.token_emb(x)
        pos = self.pos_emb(torch.arange(T, device=x.device))
        h = tok + pos

        for block in self.blocks:
            h = block(h)

        h = self.final_norm(h)
        return self.lm_head(h)


def build_model(vocab_size, block_size):
    d_model = 768
    n_layers = 8
    n_heads = 8
    d_ff = 3072

    return LegalDecoderLM(
        vocab_size=vocab_size,
        block_size=block_size,
        d_model=d_model,
        n_layers=n_layers,
        n_heads=n_heads,
        d_ff=d_ff,
    )
'''

_ARCH2_SRC = r'''class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        dropout=0.1
    ):
        super().__init__()

        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.q_proj = nn.Linear(
            d_model,
            d_model
        )

        self.k_proj = nn.Linear(
            d_model,
            d_model
        )

        self.v_proj = nn.Linear(
            d_model,
            d_model
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        query,
        key_value,
        mask=None,
        is_causal=False
    ):

        B, Tq, _ = query.shape
        _, Tk, _ = key_value.shape

        Q = self.q_proj(query)
        K = self.k_proj(key_value)
        V = self.v_proj(key_value)

        Q = Q.view(
            B,
            Tq,
            self.n_heads,
            self.d_head
        ).transpose(1, 2)

        K = K.view(
            B,
            Tk,
            self.n_heads,
            self.d_head
        ).transpose(1, 2)

        V = V.view(
            B,
            Tk,
            self.n_heads,
            self.d_head
        ).transpose(1, 2)

        dropout_p = (
            self.dropout.p
            if self.training
            else 0.0
        )

        if is_causal:

            # Native causal kernel.
            context = F.scaled_dot_product_attention(
                Q,
                K,
                V,
                attn_mask=None,
                dropout_p=dropout_p,
                is_causal=True,
            )

        else:

            if mask is not None:
                mask = mask.to(
                    device=Q.device,
                    dtype=Q.dtype
                )

            context = F.scaled_dot_product_attention(
                Q,
                K,
                V,
                attn_mask=mask,
                dropout_p=dropout_p,
                is_causal=False,
            )

        context = (
            context
            .transpose(1, 2)
            .contiguous()
            .view(
                B,
                Tq,
                self.d_model
            )
        )

        return self.out_proj(context)


# ============================================================
# 5. SINUSOIDAL POSITIONAL ENCODING
# ============================================================

def make_sinusoidal_table(
    max_len,
    d_model
):

    position = torch.arange(
        max_len,
        dtype=torch.float32
    ).unsqueeze(1)

    k = torch.arange(
        0,
        d_model,
        2,
        dtype=torch.float32
    )

    omega = 1.0 / (
        10000 ** (k / d_model)
    )

    table = torch.zeros(
        max_len,
        d_model,
        dtype=torch.float32
    )

    table[:, 0::2] = torch.sin(
        position * omega
    )

    table[:, 1::2] = torch.cos(
        position * omega
    )

    return table


# ============================================================
# 6. FEED-FORWARD
# ============================================================

class FeedForward(nn.Module):

    def __init__(
        self,
        d_model,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.linear1 = nn.Linear(
            d_model,
            d_ff
        )

        self.activation = nn.GELU()

        self.linear2 = nn.Linear(
            d_ff,
            d_model
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):

        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        x = self.dropout(x)

        return x


# ============================================================
# 7. PRE-LN ENCODER LAYER
# ============================================================

class TransformerEncoderLayerPreLN(
    nn.Module
):

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.self_attn = MultiHeadAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.ffn = FeedForward(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):

        # Pre-LN:
        # x + Attention(LN(x))

        nx = self.norm1(x)

        x = x + self.dropout(
            self.self_attn(
                nx,
                nx
            )
        )

        # Pre-LN:
        # x + FFN(LN(x))

        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ============================================================
# 8. PRE-LN ENCODER
# ============================================================

class TransformerEncoderPreLN(
    nn.Module
):

    def __init__(
        self,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        dropout=0.1
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            TransformerEncoderLayerPreLN(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

    def forward(self, x):

        for layer in self.layers:
            x = layer(x)

        return x


# ============================================================
# 9. PRE-LN DECODER LAYER
# ============================================================

class TransformerDecoderLayerPreLN(
    nn.Module
):

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.self_attn = MultiHeadAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.cross_attn = MultiHeadAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout
        )

        self.norm3 = nn.LayerNorm(
            d_model
        )

        self.ffn = FeedForward(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x,
        memory
    ):

        # ----------------------------------------------------
        # 1. Causal self-attention
        # ----------------------------------------------------

        nx = self.norm1(x)

        x = x + self.dropout(
            self.self_attn(
                nx,
                nx,
                is_causal=True
            )
        )

        # ----------------------------------------------------
        # 2. Cross-attention
        # Q = decoder
        # K/V = encoder
        # ----------------------------------------------------

        x = x + self.dropout(
            self.cross_attn(
                self.norm2(x),
                memory
            )
        )

        # ----------------------------------------------------
        # 3. FFN
        # ----------------------------------------------------

        x = x + self.ffn(
            self.norm3(x)
        )

        return x


# ============================================================
# 10. PRE-LN DECODER
# ============================================================

class TransformerDecoderPreLN(
    nn.Module
):

    def __init__(
        self,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        max_len=256,
        dropout=0.1
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            TransformerDecoderLayerPreLN(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        # Kept for diagnostics/check compatibility.
        self.register_buffer(
            "causal_mask",
            torch.triu(
                torch.full(
                    (max_len, max_len),
                    float("-inf")
                ),
                diagonal=1
            ),
            persistent=False
        )

    def forward(
        self,
        x,
        memory
    ):

        for layer in self.layers:

            x = layer(
                x,
                memory
            )

        return x


# ============================================================
# 11. COMPLETE PRE-LN MODEL
# ============================================================

class TransformerModelPreLN(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        max_src_len,
        max_tgt_len,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        dropout=0.1
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

        # Shared embedding.
        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # Fixed source positions.
        self.register_buffer(
            "src_pos_encoding",
            make_sinusoidal_table(
                max_src_len,
                d_model
            ),
            persistent=False
        )

        # Fixed target positions.
        self.register_buffer(
            "tgt_pos_encoding",
            make_sinusoidal_table(
                max_tgt_len,
                d_model
            ),
            persistent=False
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.encoder = TransformerEncoderPreLN(
            d_model=d_model,
            n_heads=n_heads,
            d_ff=d_ff,
            n_layers=n_layers,
            dropout=dropout
        )

        self.decoder = TransformerDecoderPreLN(
            d_model=d_model,
            n_heads=n_heads,
            d_ff=d_ff,
            n_layers=n_layers,
            max_len=max_tgt_len,
            dropout=dropout
        )

        # Tied output head.
        self.output_projection = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        self.output_projection.weight = (
            self.token_embedding.weight
        )

        self._init_weights()

    def _init_weights(self):

        nn.init.normal_(
            self.token_embedding.weight,
            mean=0.0,
            std=0.02
        )

        for module in self.modules():

            if isinstance(
                module,
                nn.Linear
            ):

                if (
                    module.weight
                    is not self.token_embedding.weight
                ):

                    nn.init.xavier_uniform_(
                        module.weight
                    )

                if module.bias is not None:
                    nn.init.zeros_(
                        module.bias
                    )

            elif isinstance(
                module,
                nn.LayerNorm
            ):

                nn.init.ones_(
                    module.weight
                )

                nn.init.zeros_(
                    module.bias
                )

    def encode(self, src):

        _, T = src.shape

        assert T <= self.max_src_len

        x = self.token_embedding(src)

        x = x * math.sqrt(
            self.d_model
        )

        x = (
            x
            + self.src_pos_encoding[
                :T
            ].unsqueeze(0)
        )

        x = self.dropout(x)

        return self.encoder(x)

    def decode(
        self,
        tgt_in,
        memory
    ):

        _, T = tgt_in.shape

        assert T <= self.max_tgt_len

        x = self.token_embedding(
            tgt_in
        )

        x = x * math.sqrt(
            self.d_model
        )

        x = (
            x
            + self.tgt_pos_encoding[
                :T
            ].unsqueeze(0)
        )

        x = self.dropout(x)

        x = self.decoder(
            x,
            memory
        )

        return self.output_projection(x)

    def forward(
        self,
        src,
        tgt_in
    ):

        memory = self.encode(src)

        return self.decode(
            tgt_in,
            memory
        )


# ============================================================
# 12. BUILD MODEL
# ============================================================

def build_model(
    vocab_size,
    src_len,
    tgt_len
):

    assert src_len + tgt_len == 1024

    return TransformerModelPreLN(
        vocab_size=vocab_size,
        max_src_len=src_len,
        max_tgt_len=tgt_len,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        dropout=0.1
    )
'''

_ARCH3_SRC = r'''def sinusoidal_pos_emb(T, d_model, device):
    """
    Fixed sinusoidal position encoding.
    Shape: (1, T, d_model). No learned parameters.
    """
    pos = torch.arange(T, device=device, dtype=torch.float32).unsqueeze(1)

    div = torch.exp(
        torch.arange(0, d_model, 2, device=device, dtype=torch.float32)
        * (-math.log(10000.0) / d_model)
    )

    pe = torch.zeros(T, d_model, device=device, dtype=torch.float32)
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)

    return pe.unsqueeze(0)  # (1, T, d_model)


class SlidingWindowCausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size, window_size):
        super().__init__()

        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.window_size = window_size

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        # Mask rule:
        # token i may attend to token j iff j <= i and i - j < W.
        #
        # For torch.nn.functional.scaled_dot_product_attention,
        # a boolean mask means True = allowed.
        idx = torch.arange(block_size)

        causal = idx[None, :] <= idx[:, None]          # j <= i
        window = (idx[:, None] - idx[None, :]) < window_size  # i - j < W

        mask = causal & window

        self.register_buffer(
            "swa_mask",
            mask.view(1, 1, block_size, block_size),
            persistent=False,
        )

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # Slice the precomputed mask for shorter inputs.
        attn_mask = self.swa_mask[:, :, :T, :T]

        # Do NOT use is_causal=True together with an explicit mask.
        # The banded mask already includes causality.
        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=0.0,
            is_causal=False,
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)

        return self.out_proj(y)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, block_size, window_size):
        super().__init__()

        self.norm1 = nn.LayerNorm(d_model)

        self.attn = SlidingWindowCausalSelfAttention(
            d_model=d_model,
            n_heads=n_heads,
            block_size=block_size,
            window_size=window_size,
        )

        self.norm2 = nn.LayerNorm(d_model)

        self.mlp = FeedForward(d_model, d_ff)

    def forward(self, x):
        # Pre-LN residual block.
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class LegalDecoderLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        block_size,
        d_model,
        n_layers,
        n_heads,
        d_ff,
        window_size,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.block_size = block_size
        self.d_model = d_model
        self.n_layers = n_layers

        self.token_emb = nn.Embedding(vocab_size, d_model)

        # Scale token embeddings by sqrt(d_model) so they are comparable
        # in magnitude to the sinusoidal position table.
        self.embed_scale = math.sqrt(d_model)

        # Precompute sinusoidal positions once. Not a parameter.
        pos = sinusoidal_pos_emb(block_size, d_model, device="cpu")
        self.register_buffer("pos_emb", pos, persistent=False)

        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                block_size=block_size,
                window_size=window_size,
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)

        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Tied embedding and output head.
        self.lm_head.weight = self.token_emb.weight

        self.apply(self._init_weights)

        # GPT-style residual scaling for deep pre-LN stacks.
        for block in self.blocks:
            block.attn.out_proj.weight.data.mul_(1.0 / math.sqrt(2 * self.n_layers))
            block.mlp.fc2.weight.data.mul_(1.0 / math.sqrt(2 * self.n_layers))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, x):
        B, T = x.shape

        assert T <= self.block_size, (
            f"Sequence length {T} exceeds block_size {self.block_size}"
        )

        tok = self.token_emb(x) * self.embed_scale

        pos = self.pos_emb[:, :T, :].to(tok.dtype)

        h = tok + pos

        for block in self.blocks:
            h = block(h)

        h = self.final_norm(h)

        return self.lm_head(h)


def build_model(vocab_size, block_size, window_size=256):
    """
    Architecture 3 size:
      d_model = 1024
      n_layers = 16
      n_heads = 16
      d_ff = 4096
      W = 256 by default

    Parameter count with vocab_size=40000:
      242,501,632 ≈ 242.5M
    """
    d_model = 1024
    n_layers = 16
    n_heads = 16
    d_ff = 4096

    return LegalDecoderLM(
        vocab_size=vocab_size,
        block_size=block_size,
        d_model=d_model,
        n_layers=n_layers,
        n_heads=n_heads,
        d_ff=d_ff,
        window_size=window_size,
    )
'''


def _namespace(src, name):
    ns = {"math": math, "torch": torch, "nn": nn, "F": F, "__name__": f"phase1_{name}"}
    exec(compile(src, f"<phase1 {name} definition>", "exec"), ns)
    return ns


# ======================= loading =======================
def _strip_compile_prefix(sd):
    return {(k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v for k, v in sd.items()}


def load_backbone(name, ckpt_path=None, device="cuda"):
    """Returns SimpleNamespace(model, hidden, max_len, d_model, meta). Weights are loaded strict=True.

    FIX: the whole build + load is wrapped in `torch.inference_mode(False)` so that the
    returned parameters are normal (non-inference) tensors, even if this function is
    called from inside a `torch.inference_mode()` block (as `selftest` used to do).
    Otherwise the returned backbone would be unusable for Phase-2 training.
    """
    if name not in ("arch1", "arch2", "arch3"):
        raise ValueError("name must be arch1, arch2 or arch3")

    ckpt_path = ckpt_path or os.environ.get(f"CKPT_{name.upper()}") or DEFAULT_CKPTS[name]

    with torch.inference_mode(False):
        if name == "arch1":
            ns = _namespace(_ARCH1_SRC, name)
            model = ns["build_model"](VOCAB, 1024)
            max_len, d_model = 1024, 768
        elif name == "arch2":
            ns = _namespace(_ARCH2_SRC, name)
            model = ns["build_model"](VOCAB, 768, 256)
            max_len, d_model = 768, 768
        else:  # arch3
            ns = _namespace(_ARCH3_SRC, name)
            model = ns["build_model"](VOCAB, 1024, 256)
            max_len, d_model = 1024, 1024

        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        state = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
        model.load_state_dict(_strip_compile_prefix(state), strict=True)

        n_params = sum(p.numel() for p in model.parameters())
        meta = {
            "name": name,
            "ckpt": ckpt_path,
            "n_params": n_params,
            "step": ck.get("step") if isinstance(ck, dict) else None,
            "best": ck.get("best") if isinstance(ck, dict) else None,
        }
        model = model.to(device).eval()

    # Define hidden() AFTER the inference_mode(False) block so it captures the final model.
    if name == "arch1":
        def hidden(ids):
            T = ids.shape[1]
            assert T <= max_len, f"sequence length {T} > max_len {max_len}"
            h = model.token_emb(ids) + model.pos_emb(torch.arange(T, device=ids.device))
            for block in model.blocks:
                h = block(h)
            return model.final_norm(h)

    elif name == "arch2":
        def hidden(ids):
            T = ids.shape[1]
            assert T <= max_len, f"sequence length {T} > max_len {max_len}"
            return model.encode(ids)               # raw encoder output; there is no final norm

    else:  # arch3
        def hidden(ids):
            T = ids.shape[1]
            assert T <= max_len, f"sequence length {T} > max_len {max_len}"
            tok = model.token_emb(ids) * model.embed_scale
            pos = model.pos_emb[:, :T, :].to(tok.dtype)
            h = tok + pos
            for block in model.blocks:
                h = block(h)
            return model.final_norm(h)

    return SimpleNamespace(model=model, hidden=hidden, max_len=max_len, d_model=d_model, meta=meta)


# ======================= self-test =======================
@torch.no_grad()
def selftest(name, ckpt_path=None, device="cuda"):
    """Run sanity checks on one Phase-1 backbone.

    NOTE: we deliberately use `torch.no_grad()` here (not `torch.inference_mode()`),
    because `load_backbone` builds a model whose parameters must remain ordinary
    tensors so they can be reused for Phase-2 training.
    """
    print("\n" + "=" * 72 + f"\n{name}\n" + "=" * 72)
    bb = load_backbone(name, ckpt_path, device)
    m, meta = bb.model, bb.meta
    print(f"checkpoint: {meta['ckpt']}\n  step={meta['step']}  best={meta['best']}")
    print(f"  parameters: {meta['n_params']:,}  (expected {EXPECTED_PARAMS[name]:,})")
    assert meta["n_params"] == EXPECTED_PARAMS[name], "parameter count differs from the Phase-1 model"
    print("  strict load: PASS | parameter count: PASS")

    g = torch.Generator(device="cpu").manual_seed(0)
    T = 512
    x = torch.randint(0, VOCAB, (2, T), generator=g).to(device)
    h = bb.hidden(x).float()
    assert h.shape == (2, T, bb.d_model) and torch.isfinite(h).all()
    print(f"  hidden shape {tuple(h.shape)}: PASS")

    if name in ("arch1", "arch3"):
        diff = (m.lm_head(bb.hidden(x)).float() - m(x).float()).abs().max().item()
        print(f"  parity  lm_head(hidden(x)) vs model(x): max|diff| = {diff:.2e}")
        assert diff < 1e-3, "hidden() does not reproduce the model's own forward pass"

        t = T // 2
        x2 = x.clone()
        x2[:, t:] = torch.randint(0, VOCAB, x2[:, t:].shape, generator=g).to(device)
        leak = (bb.hidden(x).float()[:, :t] - bb.hidden(x2).float()[:, :t]).abs().max().item()
        print(f"  causality: changing tokens >= {t} moves earlier states by {leak:.2e} (must be ~0)")
        assert leak < 1e-3

    else:  # arch2
        x2 = x.clone()
        x2[:, -1] = (x2[:, -1] + 1) % VOCAB
        early = (bb.hidden(x).float()[:, 0] - bb.hidden(x2).float()[:, 0]).abs().max().item()
        print(f"  bidirectional encoder: changing ONLY the last token moves position-0 state by {early:.2e} (expected > 0)")
        assert early > 0
        print("  (no padding mask exists: padded positions are attended to like real tokens)")

    n = h.norm(dim=-1)
    print(f"  activation scale: per-token L2 norm mean {n.mean():.2f}, std {n.std():.2f}, max {n.max():.2f} | "
          f"max |value| {h.abs().max():.2f} | per-dimension std (median) {h.std(dim=(0, 1)).median():.3f}")
    return bb


def selftest_all(device="cuda"):
    for name in ("arch1", "arch2", "arch3"):
        try:
            selftest(name, None, device)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:                                   # keep going; show which one failed
            print(f"\nFAILED {name}: {type(e).__name__}: {e}")


if __name__ == "__main__":
    selftest_all("cuda" if torch.cuda.is_available() else "cpu")


arch1
checkpoint: /kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt
  step=10000  best=4.100625859101613
  parameters: 88,210,944  (expected 88,210,944)
  strict load: PASS | parameter count: PASS
  hidden shape (2, 512, 768): PASS
  parity  lm_head(hidden(x)) vs model(x): max|diff| = 0.00e+00
  causality: changing tokens >= 256 moves earlier states by 0.00e+00 (must be ~0)
  activation scale: per-token L2 norm mean 22.02, std 0.00, max 22.04 | max |value| 2.09 | per-dimension std (median) 0.031

arch2
checkpoint: /kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt
  step=10000  best=2.906713534593582
  parameters: 129,957,888  (expected 129,957,888)
  strict load: PASS | parameter count: PASS
  hidden shape (2, 512, 768): PASS
  bidirectional encoder: changing ONLY the last token moves position-0 state by 6.25e-03 (expected > 0)
  (no padding mask exists: padded positions are attended to like 

In [3]:
# ============================================================
# PHASE 2 — EXPERIMENT 3: ARCH1 + LoRA FOR LEGAL NER
# ============================================================
# Paste this entire cell AFTER the notebook's Phase-2 data/BIO
# preparation cells. It uses the existing notebook globals:
#   tokenizer, correct_offsets, train_docs, val_docs,
#   ENTITY_LABELS, TAG2ID, ID2TAG
# and loads the saved 1024-token window tensors.
#
# It DOES NOT use the official test fold.
# It does NOT change the existing Arch4 experiment files.
# ============================================================

import os
import json
import math
import time
import random
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# ------------------------------------------------------------
# 1. Configuration and preflight
# ------------------------------------------------------------

CFG = {
    "EXPERIMENT": "phase2_arch1_lora",
    "SEED": 42,
    "VOCAB_SIZE": 40000,
    "MAX_LEN": 1024,
    "NUM_CLASSES": 25,
    "IGNORE_INDEX": -100,
    "EPOCHS": 5,
    "BATCH_SIZE": 1,
    "GRAD_ACCUM": 8,
    "LORA_RANK": 8,
    "LORA_ALPHA": 16,
    "LORA_DROPOUT": 0.05,
    "LORA_LR": 1e-4,
    "HEAD_LR": 2e-4,
    "WEIGHT_DECAY": 0.01,
    "MAX_GRAD_NORM": 1.0,
}

ARCH1_CKPT = (
    "/kaggle/input/models/achiver7/arch1-modeldecoder/"
    "transformers/default/1/run_arch1_real/best.pt"
)

DATA_DIR_CANDIDATES = [
    Path("/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data"),
    Path("/kaggle/working/phase2_arch4_data"),
]

NER_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER"),
    Path("/kaggle/working/ner_data/Phase_2_LNER"),
    Path("/kaggle/input/ner_data/Phase_2_LNER"),
]

TOKENIZER_PATH_CANDIDATES = [
    Path("/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"),
    Path("/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/tokenizer.json"),
    Path("/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokens/tokenizer.json"),
]

RUN_DIR = Path("/kaggle/working/phase2_arch1_lora")
BEST_PATH = RUN_DIR / "best_arch1_lora_strict_macro_f1.pt"
HISTORY_PATH = RUN_DIR / "training_history.json"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

# ------------------------------------------------------------
# Resolve saved Phase-2 windows and split manifest first.
# The split manifest, not a newly generated random split, is
# authoritative so this experiment uses the existing 56/14 split.
# ------------------------------------------------------------

train_data_path = val_data_path = manifest_path = None
for candidate in DATA_DIR_CANDIDATES:
    tp = candidate / "train_windows.pt"
    vp = candidate / "validation_windows.pt"
    mp = candidate / "split_manifest.json"
    if tp.is_file() and vp.is_file() and mp.is_file():
        train_data_path, val_data_path, manifest_path = tp, vp, mp
        break

if train_data_path is None:
    raise FileNotFoundError(
        "Could not find train_windows.pt, validation_windows.pt and split_manifest.json. "
        "Checked: " + ", ".join(str(p) for p in DATA_DIR_CANDIDATES)
    )

ner_root = next(
    (p for p in NER_ROOT_CANDIDATES
     if all((p / rel).is_file() for rel in [
         "Train/fold1-text.json", "Train/fold1-gold.json",
         "Train/fold2-text.json", "Train/fold2-gold.json",
         "Train/labels.txt"
     ])),
    None,
)
if ner_root is None:
    raise FileNotFoundError(
        "Could not find the Phase-2 source text/gold files and labels.txt. Checked: "
        + ", ".join(str(p) for p in NER_ROOT_CANDIDATES)
    )

from tokenizers import Tokenizer

TOKENIZER_PATH = next(
    (p for p in TOKENIZER_PATH_CANDIDATES if p.is_file()),
    None,
)
if TOKENIZER_PATH is None:
    raise FileNotFoundError(
        "Could not find the Phase-1 tokenizer.json. Checked: "
        + ", ".join(str(p) for p in TOKENIZER_PATH_CANDIDATES)
    )

# Build all notebook variables this script needs. This makes the
# experiment self-contained and fixes the Missing notebook variables
# error without requiring earlier preprocessing cells to be executed.
with open(ner_root / "Train/labels.txt", "r", encoding="utf-8") as f:
    ENTITY_LABELS = [line.strip() for line in f if line.strip()]

with open(ner_root / "Train/fold1-text.json", "r", encoding="utf-8") as f:
    fold1_text = json.load(f)
with open(ner_root / "Train/fold1-gold.json", "r", encoding="utf-8") as f:
    fold1_gold = json.load(f)
with open(ner_root / "Train/fold2-text.json", "r", encoding="utf-8") as f:
    fold2_text = json.load(f)
with open(ner_root / "Train/fold2-gold.json", "r", encoding="utf-8") as f:
    fold2_gold = json.load(f)

# Normalize IDs to strings, matching saved window metadata.
def _normalize_text_map(raw, name):
    out = {}
    for key, value in raw.items():
        if isinstance(value, dict):
            value = value.get("text")
        if not isinstance(value, str):
            raise TypeError(f"{name}[{key!r}] is not a text string")
        out[str(key)] = value
    return out

def _normalize_gold_map(raw, name):
    out = {str(key): value for key, value in raw.items()}
    for key, entities in out.items():
        if not isinstance(entities, list):
            raise TypeError(f"{name}[{key!r}] must be a list of entity spans")
    return out

fold1_text = _normalize_text_map(fold1_text, "fold1_text")
fold2_text = _normalize_text_map(fold2_text, "fold2_text")
fold1_gold = _normalize_gold_map(fold1_gold, "fold1_gold")
fold2_gold = _normalize_gold_map(fold2_gold, "fold2_gold")

if set(fold1_text) != set(fold1_gold):
    raise ValueError("Fold 1 text IDs and gold IDs do not match")
if set(fold2_text) != set(fold2_gold):
    raise ValueError("Fold 2 text IDs and gold IDs do not match")
if set(fold1_text) & set(fold2_text):
    raise ValueError("Duplicate document IDs exist across Fold 1 and Fold 2")

all_docs = {}
for text_map, gold_map in ((fold1_text, fold1_gold), (fold2_text, fold2_gold)):
    for doc_id, text in text_map.items():
        all_docs[doc_id] = {"text": text, "entities": gold_map[doc_id]}

if len(all_docs) != 70:
    raise ValueError(f"Expected 70 labelled documents, found {len(all_docs)}")

# Same offset correction used by the existing Arch4 preprocessing.
def correct_offsets(text, raw_offsets):
    corrected = []
    for start, end in raw_offsets:
        start, end = int(start), int(end)
        if not (0 <= start <= end <= len(text)):
            raise ValueError(
                f"Invalid tokenizer offset {(start, end)} for text length {len(text)}"
            )
        while start < end and text[start].isspace():
            start += 1
        corrected.append((start, end))
    return corrected

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

with open(manifest_path, "r", encoding="utf-8") as f:
    split_manifest = json.load(f)

if "train_doc_ids" not in split_manifest or "validation_doc_ids" not in split_manifest:
    raise KeyError(f"Split manifest is missing train/validation IDs: {manifest_path}")

train_ids = [str(x) for x in split_manifest["train_doc_ids"]]
val_ids = [str(x) for x in split_manifest["validation_doc_ids"]]
if len(train_ids) != 56 or len(val_ids) != 14:
    raise ValueError(
        f"Expected manifest split 56/14 documents; got {len(train_ids)}/{len(val_ids)}"
    )
if len(set(train_ids)) != len(train_ids) or len(set(val_ids)) != len(val_ids):
    raise ValueError("Duplicate document IDs in split manifest")
if set(train_ids) & set(val_ids):
    raise ValueError("Train and validation document IDs overlap in split manifest")
if set(train_ids) | set(val_ids) != set(all_docs):
    missing = (set(train_ids) | set(val_ids)) ^ set(all_docs)
    raise ValueError(
        "The saved split manifest does not cover exactly the 70 source documents. "
        f"Mismatched IDs (first 10): {sorted(missing)[:10]}"
    )

train_docs = {doc_id: all_docs[doc_id] for doc_id in train_ids}
val_docs = {doc_id: all_docs[doc_id] for doc_id in val_ids}

test_ids = set(str(x) for x in split_manifest.get("test_doc_ids", []))
if test_ids and (test_ids & (set(train_docs) | set(val_docs))):
    raise ValueError("The saved split manifest overlaps train/validation with official test IDs")

BIO_TAGS = (
    ["O"]
    + [f"B-{label}" for label in ENTITY_LABELS]
    + [f"I-{label}" for label in ENTITY_LABELS]
)
TAG2ID = {tag: idx for idx, tag in enumerate(BIO_TAGS)}
ID2TAG = {idx: tag for tag, idx in TAG2ID.items()}

if len(ENTITY_LABELS) != 12 or len(TAG2ID) != CFG["NUM_CLASSES"] or len(ID2TAG) != CFG["NUM_CLASSES"]:
    raise ValueError(
        f"Expected 12 entity types and 25 BIO classes; got {len(ENTITY_LABELS)} and {len(TAG2ID)}"
    )
if TAG2ID.get("O") != 0:
    raise ValueError("BIO mapping mismatch: O must map to class 0")

# Verify that saved window labels use the same class mapping.
manifest_tags = split_manifest.get("bio_tags")
if manifest_tags is not None and list(manifest_tags) != BIO_TAGS:
    raise ValueError("BIO tag order in split_manifest.json differs from the current mapping")

if not os.path.isfile(ARCH1_CKPT):
    raise FileNotFoundError(f"Arch1 checkpoint not found: {ARCH1_CKPT}")

# Don't silently overwrite an earlier run's best model.
if BEST_PATH.exists():
    raise FileExistsError(
        f"A checkpoint already exists at {BEST_PATH}. "
        "To preserve it, change RUN_DIR to a new experiment directory "
        "before rerunning."
    )

random.seed(CFG["SEED"])
torch.manual_seed(CFG["SEED"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG["SEED"])

RUN_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 76)
print("PHASE 2 — ARCH1 + LoRA NER")
print("=" * 76)
print("Device:", DEVICE)
if USE_AMP:
    print("GPU:", torch.cuda.get_device_name(0))
print("Arch1 checkpoint:", ARCH1_CKPT)
print("Training windows:", train_data_path)
print("Validation windows:", val_data_path)
print("Output directory:", RUN_DIR)

# ------------------------------------------------------------
# 2. Load and validate the existing token windows
# ------------------------------------------------------------

train_data = torch.load(train_data_path, map_location="cpu", weights_only=False)
val_data = torch.load(val_data_path, map_location="cpu", weights_only=False)

for split_name, data in [("TRAIN", train_data), ("VALIDATION", val_data)]:
    x, y, metadata = data["input_ids"], data["labels"], data["metadata"]
    print(
        f"{split_name}: windows={len(metadata)}, shape={tuple(x.shape)}, "
        f"supervised_tokens={int(y.ne(CFG['IGNORE_INDEX']).sum())}"
    )
    assert x.ndim == 2 and y.ndim == 2 and x.shape == y.shape
    assert x.shape[1] == CFG["MAX_LEN"]
    assert x.dtype == torch.long and y.dtype == torch.long
    assert len(metadata) == x.shape[0]
    assert int(x.min()) >= 0 and int(x.max()) < CFG["VOCAB_SIZE"]
    valid_labels = y[y != CFG["IGNORE_INDEX"]]
    assert valid_labels.numel() > 0
    assert int(valid_labels.min()) >= 0 and int(valid_labels.max()) < CFG["NUM_CLASSES"]
    assert (y.ne(CFG["IGNORE_INDEX"]).sum(dim=1) > 0).all()

# Confirm the cached windows actually correspond to the same document
# partition recorded in split_manifest.json. Never silently evaluate the
# wrong fold or accidentally mix train and validation documents.
train_window_doc_ids = {str(m["doc_id"]) for m in train_data["metadata"]}
val_window_doc_ids = {str(m["doc_id"]) for m in val_data["metadata"]}
if train_window_doc_ids != set(train_ids):
    raise ValueError(
        "Training-window document IDs do not match split_manifest.json. "
        f"Window IDs={len(train_window_doc_ids)}, manifest IDs={len(train_ids)}"
    )
if val_window_doc_ids != set(val_ids):
    raise ValueError(
        "Validation-window document IDs do not match split_manifest.json. "
        f"Window IDs={len(val_window_doc_ids)}, manifest IDs={len(val_ids)}"
    )
if train_window_doc_ids & val_window_doc_ids:
    raise ValueError("Train and validation windows contain overlapping document IDs")

print("Self-contained data setup: PASS")
print("NER root:", ner_root)
print("Tokenizer:", TOKENIZER_PATH)
print("Split manifest:", manifest_path)
print("Documents: train=", len(train_docs), "validation=", len(val_docs))

train_dataset = TensorDataset(train_data["input_ids"], train_data["labels"])
generator = torch.Generator().manual_seed(CFG["SEED"])
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["BATCH_SIZE"],
    shuffle=True,
    generator=generator,
    num_workers=0,
    pin_memory=USE_AMP,
)

# ------------------------------------------------------------
# 3. Exact Arch1 Phase-1 model definition
#    (matches the verified Arch1 checkpoint architecture)
# ------------------------------------------------------------

class Arch1CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("mask", mask)  # Preserve the Phase-1 checkpoint buffer key for strict loading.

    def forward(self, x):
        batch, seq_len, channels = x.shape
        q = self.q_proj(x).view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = attn.masked_fill(self.mask[:, :, :seq_len, :seq_len] == 0, float("-inf"))
        attn = F.softmax(attn, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(batch, seq_len, channels)
        return self.out_proj(out)


class Arch1FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class Arch1TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, block_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = Arch1CausalSelfAttention(d_model, n_heads, block_size)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = Arch1FeedForward(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class Arch1LegalDecoderLM(nn.Module):
    def __init__(self, vocab_size=40000, block_size=1024, d_model=768, n_layers=8, n_heads=8, d_ff=3072):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([
            Arch1TransformerBlock(d_model, n_heads, d_ff, block_size)
            for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

    def forward_hidden(self, input_ids):
        batch, seq_len = input_ids.shape
        if seq_len > self.block_size:
            raise ValueError(f"Sequence length {seq_len} exceeds {self.block_size}")
        pos_ids = torch.arange(seq_len, device=input_ids.device)
        x = self.token_emb(input_ids) + self.pos_emb(pos_ids)
        for block in self.blocks:
            x = block(x)
        return self.final_norm(x)

    def forward(self, input_ids):
        return self.lm_head(self.forward_hidden(input_ids))


# Load Phase-1 checkpoint strictly before inserting adapters.
arch1 = Arch1LegalDecoderLM(
    vocab_size=CFG["VOCAB_SIZE"],
    block_size=CFG["MAX_LEN"],
)
raw_ckpt = torch.load(ARCH1_CKPT, map_location="cpu", weights_only=False)
ARCH1_PHASE1_STEP = raw_ckpt.get("step") if isinstance(raw_ckpt, dict) else None
ARCH1_PHASE1_BEST = raw_ckpt.get("best") if isinstance(raw_ckpt, dict) else None
if isinstance(raw_ckpt, dict) and isinstance(raw_ckpt.get("model"), dict):
    arch1_state = raw_ckpt["model"]
elif isinstance(raw_ckpt, dict) and isinstance(raw_ckpt.get("state_dict"), dict):
    arch1_state = raw_ckpt["state_dict"]
else:
    arch1_state = raw_ckpt
arch1_state = {
    (key[len("_orig_mod."):] if key.startswith("_orig_mod.") else key): value
    for key, value in arch1_state.items()
}
arch1.load_state_dict(arch1_state, strict=True)
del raw_ckpt, arch1_state

actual_params = sum(p.numel() for p in arch1.parameters())
assert actual_params == 88_210_944, f"Unexpected Arch1 parameter count: {actual_params:,}"
print("Arch1 strict checkpoint load: PASS | parameters:", f"{actual_params:,}")
print("Phase-1 checkpoint step:", ARCH1_PHASE1_STEP)
print("Phase-1 best validation loss:", ARCH1_PHASE1_BEST)

# Move the unmodified backbone to the target device and record a
# baseline hidden-state tensor for the LoRA identity-initialization test.
arch1 = arch1.to(DEVICE).eval()
with torch.inference_mode():
    identity_test_ids = train_data["input_ids"][:1, :128].to(DEVICE)
    hidden_before_lora = arch1.forward_hidden(identity_test_ids).float().cpu()
del identity_test_ids

# ------------------------------------------------------------
# 4. Additive LoRA to six linear projections per transformer block
# ------------------------------------------------------------

class LoRALinear(nn.Module):
    """Frozen linear layer plus additive low-rank update: W0*x + alpha/r * B(A(dropout(x)))."""
    def __init__(self, base_layer, rank=8, alpha=16, dropout=0.05):
        super().__init__()
        if not isinstance(base_layer, nn.Linear):
            raise TypeError("LoRALinear requires an nn.Linear base layer")
        self.base = base_layer
        for p in self.base.parameters():
            p.requires_grad = False
        self.rank = int(rank)
        self.alpha = float(alpha)
        self.scaling = self.alpha / self.rank
        self.dropout = nn.Dropout(dropout)
        self.lora_A = nn.Linear(base_layer.in_features, self.rank, bias=False)
        self.lora_B = nn.Linear(self.rank, base_layer.out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)  # exact identity at initialization

    def forward(self, x):
        return self.base(x) + self.scaling * self.lora_B(self.lora_A(self.dropout(x)))


def replace_linear(parent, attribute, cfg):
    original = getattr(parent, attribute)
    setattr(parent, attribute, LoRALinear(
        original,
        rank=cfg["LORA_RANK"],
        alpha=cfg["LORA_ALPHA"],
        dropout=cfg["LORA_DROPOUT"],
    ))

# Freeze the original backbone first. Newly created LoRA A/B weights remain trainable.
for parameter in arch1.parameters():
    parameter.requires_grad = False

target_names = [
    ("attn", "q_proj"),
    ("attn", "k_proj"),
    ("attn", "v_proj"),
    ("attn", "out_proj"),
    ("mlp", "fc1"),
    ("mlp", "fc2"),
]

lora_module_count = 0
for block in arch1.blocks:
    for parent_name, layer_name in target_names:
        parent = getattr(block, parent_name)
        replace_linear(parent, layer_name, CFG)
        lora_module_count += 1

assert lora_module_count == 8 * 6, f"Expected 48 LoRA modules; found {lora_module_count}"
arch1 = arch1.to(DEVICE)

bio_head = nn.Linear(arch1.d_model, CFG["NUM_CLASSES"]).to(DEVICE)

# Verify LoRA's zero-B initialization preserves the pretrained hidden states.
arch1.eval()
bio_head.eval()
with torch.inference_mode():
    identity_test_ids = train_data["input_ids"][:1, :128].to(DEVICE)
    hidden_after_lora = arch1.forward_hidden(identity_test_ids).float().cpu()
    test_ids = train_data["input_ids"][:1].to(DEVICE)
    hidden = arch1.forward_hidden(test_ids)
    test_logits = bio_head(hidden)
identity_error = (hidden_before_lora - hidden_after_lora).abs().max().item()
assert identity_error < 1e-5, f"LoRA initialization changed pretrained states: max diff={identity_error}"
assert hidden.shape == (1, CFG["MAX_LEN"], 768)
assert test_logits.shape == (1, CFG["MAX_LEN"], CFG["NUM_CLASSES"])
assert torch.isfinite(test_logits).all()
del identity_test_ids, hidden_before_lora, hidden_after_lora, test_ids, hidden, test_logits
if USE_AMP:
    torch.cuda.empty_cache()
print("LoRA insertion:", lora_module_count, "modules | forward preflight: PASS")
print("Identity initialization max hidden-state difference:", f"{identity_error:.3e}")

# Trainable params are only LoRA A/B matrices plus the NER head.
trainable_backbone = [p for p in arch1.parameters() if p.requires_grad]
for p in bio_head.parameters():
    p.requires_grad = True
trainable_head = list(bio_head.parameters())
trainable_total = sum(p.numel() for p in trainable_backbone + trainable_head)
print("Trainable LoRA parameters:", f"{sum(p.numel() for p in trainable_backbone):,}")
print("Trainable head parameters:", f"{sum(p.numel() for p in trainable_head):,}")
print("Total trainable parameters:", f"{trainable_total:,}", f"({100 * trainable_total / actual_params:.3f}% of backbone parameters)")

optimizer = torch.optim.AdamW(
    [
        {"params": trainable_backbone, "lr": CFG["LORA_LR"]},
        {"params": trainable_head, "lr": CFG["HEAD_LR"]},
    ],
    weight_decay=CFG["WEIGHT_DECAY"],
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

# ------------------------------------------------------------
# 5. Strict document-level character-span evaluator
# ------------------------------------------------------------

@torch.inference_mode()
def evaluate_arch1_lora():
    prev_model_mode = arch1.training
    prev_head_mode = bio_head.training
    arch1.eval()
    bio_head.eval()

    val_by_string_id = {str(key): item for key, item in val_docs.items()}
    offsets_by_doc = {}
    pred_tags_by_doc = {}
    ownership_by_doc = {}

    for doc_id, item in val_by_string_id.items():
        encoding = tokenizer.encode(item["text"])
        offsets = correct_offsets(item["text"], encoding.offsets)
        offsets_by_doc[doc_id] = offsets
        pred_tags_by_doc[doc_id] = [0] * len(offsets)
        ownership_by_doc[doc_id] = [False] * len(offsets)

    try:
        for window_idx, meta in enumerate(val_data["metadata"]):
            doc_id = str(meta["doc_id"])
            if doc_id not in offsets_by_doc:
                raise KeyError(f"Validation metadata references unknown doc ID: {doc_id}")
            input_ids = val_data["input_ids"][window_idx:window_idx + 1].to(DEVICE)

            if USE_AMP:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    hidden_states = arch1.forward_hidden(input_ids)
                    logits = bio_head(hidden_states)
            else:
                hidden_states = arch1.forward_hidden(input_ids)
                logits = bio_head(hidden_states)

            pred_ids = logits.float().argmax(dim=-1)[0].cpu().tolist()
            window_start = int(meta["window_start"])
            valid_length = int(meta["valid_length"])
            owner_local_start = int(meta["owner_local_start"])

            for local_idx in range(owner_local_start, valid_length):
                global_idx = window_start + local_idx
                if global_idx >= len(pred_tags_by_doc[doc_id]):
                    raise IndexError(f"Token index outside document {doc_id}: {global_idx}")
                if ownership_by_doc[doc_id][global_idx]:
                    raise AssertionError(f"Duplicate prediction ownership: {doc_id}/{global_idx}")
                pred_tags_by_doc[doc_id][global_idx] = int(pred_ids[local_idx])
                ownership_by_doc[doc_id][global_idx] = True

        for doc_id, ownership in ownership_by_doc.items():
            if not all(ownership):
                raise AssertionError(f"Missing token predictions in {doc_id}: {ownership.count(False)}")

        gold_spans = Counter()
        pred_spans = Counter()

        for doc_id, item in val_by_string_id.items():
            offsets = offsets_by_doc[doc_id]
            tag_ids = pred_tags_by_doc[doc_id]
            if len(offsets) != len(tag_ids):
                raise AssertionError(f"Offset/tag length mismatch in document {doc_id}")

            for entity in item["entities"]:
                gold_spans[(doc_id, int(entity[0]), int(entity[1]), entity[2])] += 1

            readable_tags = []
            for tag_id, (start, end) in zip(tag_ids, offsets):
                readable_tags.append("O" if start >= end else ID2TAG[int(tag_id)])

            i = 0
            while i < len(readable_tags):
                tag = readable_tags[i]
                if tag == "O":
                    i += 1
                    continue
                if tag.startswith("B-") or tag.startswith("I-"):
                    label = tag[2:]
                else:
                    i += 1
                    continue

                start_char = offsets[i][0]
                j = i + 1
                while j < len(readable_tags) and readable_tags[j] == f"I-{label}":
                    j += 1
                end_char = offsets[j - 1][1]
                if start_char < end_char:
                    pred_spans[(doc_id, start_char, end_char, label)] += 1
                i = j

        matched = gold_spans & pred_spans
        tp_by_label = Counter()
        for span, count in matched.items():
            tp_by_label[span[3]] += count

        per_label = {}
        for label in ENTITY_LABELS:
            gold_count = sum(n for span, n in gold_spans.items() if span[3] == label)
            pred_count = sum(n for span, n in pred_spans.items() if span[3] == label)
            tp = tp_by_label[label]
            fp, fn = pred_count - tp, gold_count - tp
            precision = tp / (tp + fp) if tp + fp else 0.0
            recall = tp / (tp + fn) if tp + fn else 0.0
            f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
            per_label[label] = {
                "gold": int(gold_count), "predicted": int(pred_count),
                "tp": int(tp), "fp": int(fp), "fn": int(fn),
                "precision": precision, "recall": recall, "f1": f1,
            }

        return {
            "strict_macro_precision": sum(per_label[x]["precision"] for x in ENTITY_LABELS) / len(ENTITY_LABELS),
            "strict_macro_recall": sum(per_label[x]["recall"] for x in ENTITY_LABELS) / len(ENTITY_LABELS),
            "strict_macro_f1": sum(per_label[x]["f1"] for x in ENTITY_LABELS) / len(ENTITY_LABELS),
            "gold_entities": int(sum(gold_spans.values())),
            "predicted_entities": int(sum(pred_spans.values())),
            "exact_matches": int(sum(matched.values())),
            "per_label": per_label,
        }
    finally:
        arch1.train(prev_model_mode)
        bio_head.train(prev_head_mode)

# ------------------------------------------------------------
# 6. Training loop with atomic best-checkpoint/history saves
# ------------------------------------------------------------

def atomic_json_save(obj, path):
    tmp_path = str(path) + ".tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    os.replace(tmp_path, path)


def save_best(epoch, metrics, mean_train_loss, elapsed_seconds):
    ckpt = {
        "experiment": CFG["EXPERIMENT"],
        "epoch": int(epoch),
        "arch1_checkpoint": ARCH1_CKPT,
        "phase1_step": ARCH1_PHASE1_STEP,
        "phase1_best_validation_loss": ARCH1_PHASE1_BEST,
        "model": arch1.state_dict(),
        "bio_head": bio_head.state_dict(),
        "metrics": metrics,
        "mean_train_loss": float(mean_train_loss),
        "config": CFG,
        "tag2id": TAG2ID,
        "trainable_parameters": int(trainable_total),
        "backbone_parameters": int(actual_params),
        "lora_modules": int(lora_module_count),
        "elapsed_seconds": float(elapsed_seconds),
    }
    tmp_path = BEST_PATH.with_suffix(".tmp")
    torch.save(ckpt, tmp_path)
    os.replace(tmp_path, BEST_PATH)

history = []
best_f1 = -1.0
start_time = time.time()
if USE_AMP:
    torch.cuda.reset_peak_memory_stats()

for epoch in range(1, CFG["EPOCHS"] + 1):
    arch1.train()
    bio_head.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    batches_seen = 0

    for batch_idx, (input_ids, labels) in enumerate(train_loader):
        input_ids = input_ids.to(DEVICE, non_blocking=USE_AMP)
        labels = labels.to(DEVICE, non_blocking=USE_AMP)

        group_start = (batch_idx // CFG["GRAD_ACCUM"]) * CFG["GRAD_ACCUM"]
        group_size = min(CFG["GRAD_ACCUM"], len(train_loader) - group_start)

        if USE_AMP:
            amp_context = torch.autocast(device_type="cuda", dtype=torch.float16)
        else:
            from contextlib import nullcontext
            amp_context = nullcontext()

        with amp_context:
            hidden_states = arch1.forward_hidden(input_ids)
            logits = bio_head(hidden_states)
            loss = F.cross_entropy(
                logits.reshape(-1, CFG["NUM_CLASSES"]).float(),
                labels.reshape(-1),
                ignore_index=CFG["IGNORE_INDEX"],
            )

        if not torch.isfinite(loss).item():
            raise RuntimeError(f"Non-finite loss at epoch={epoch}, batch={batch_idx}")

        running_loss += float(loss.detach())
        batches_seen += 1
        scaler.scale(loss / group_size).backward()

        should_update = ((batch_idx + 1) % CFG["GRAD_ACCUM"] == 0) or ((batch_idx + 1) == len(train_loader))
        if should_update:
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(
                [p for p in arch1.parameters() if p.requires_grad] + list(bio_head.parameters()),
                CFG["MAX_GRAD_NORM"],
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        if batch_idx == 0 or (batch_idx + 1) % 50 == 0 or (batch_idx + 1) == len(train_loader):
            print(
                f"Epoch {epoch}/{CFG['EPOCHS']} | "
                f"batch {batch_idx + 1}/{len(train_loader)} | "
                f"loss={float(loss.detach()):.4f}"
            )

    mean_train_loss = running_loss / max(batches_seen, 1)
    metrics = evaluate_arch1_lora()
    elapsed = time.time() - start_time
    peak_gpu_gb = torch.cuda.max_memory_allocated() / 1e9 if USE_AMP else 0.0

    record = {
        "epoch": epoch,
        "mean_train_loss": mean_train_loss,
        "strict_macro_precision": metrics["strict_macro_precision"],
        "strict_macro_recall": metrics["strict_macro_recall"],
        "strict_macro_f1": metrics["strict_macro_f1"],
        "gold_entities": metrics["gold_entities"],
        "predicted_entities": metrics["predicted_entities"],
        "exact_matches": metrics["exact_matches"],
        "trainable_parameters": trainable_total,
        "backbone_parameters": actual_params,
        "trainable_fraction_percent": 100 * trainable_total / actual_params,
        "elapsed_seconds": elapsed,
        "peak_gpu_memory_gb": peak_gpu_gb,
        "per_label": metrics["per_label"],
    }
    history.append(record)
    atomic_json_save(history, HISTORY_PATH)

    print("\n" + "=" * 76)
    print(f"ARCH1 + LoRA — EPOCH {epoch}")
    print(f"Mean training loss: {mean_train_loss:.6f}")
    print(f"Strict Macro-Precision: {metrics['strict_macro_precision']:.6f}")
    print(f"Strict Macro-Recall:    {metrics['strict_macro_recall']:.6f}")
    print(f"Strict Macro-F1:        {metrics['strict_macro_f1']:.6f}")
    print("Gold entities:", metrics["gold_entities"])
    print("Predicted entities:", metrics["predicted_entities"])
    print("Exact matches:", metrics["exact_matches"])
    print("Trainable parameters:", f"{trainable_total:,}")
    print("Trainable fraction:", f"{100 * trainable_total / actual_params:.3f}%")
    if USE_AMP:
        print("Peak GPU memory (GB):", f"{peak_gpu_gb:.3f}")
    print("Elapsed minutes:", f"{elapsed / 60:.2f}")

    if metrics["strict_macro_f1"] > best_f1:
        best_f1 = metrics["strict_macro_f1"]
        save_best(epoch, metrics, mean_train_loss, elapsed)
        print("NEW BEST CHECKPOINT:", BEST_PATH)
    print("=" * 76 + "\n")

# ------------------------------------------------------------
# 7. Restore and re-evaluate the best checkpoint
# ------------------------------------------------------------

if not BEST_PATH.is_file():
    raise RuntimeError("Training ended without saving a best checkpoint.")

best_ckpt = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
arch1.load_state_dict(best_ckpt["model"], strict=True)
bio_head.load_state_dict(best_ckpt["bio_head"], strict=True)
arch1.to(DEVICE)
bio_head.to(DEVICE)

final_metrics = evaluate_arch1_lora()

print("\n" + "=" * 76)
print("ARCH1 + LoRA — BEST CHECKPOINT RE-EVALUATION")
print("=" * 76)
print("Saved epoch:", best_ckpt["epoch"])
print("Stored Strict Macro-F1:", f"{best_ckpt['metrics']['strict_macro_f1']:.6f}")
print("Re-evaluated Strict Macro-F1:", f"{final_metrics['strict_macro_f1']:.6f}")
print("Re-evaluated Precision:", f"{final_metrics['strict_macro_precision']:.6f}")
print("Re-evaluated Recall:", f"{final_metrics['strict_macro_recall']:.6f}")
print("Trainable parameters:", f"{trainable_total:,}")
print("Best checkpoint:", BEST_PATH)
print("Training history:", HISTORY_PATH)
print("Official test fold: NOT USED")

assert abs(final_metrics["strict_macro_f1"] - best_ckpt["metrics"]["strict_macro_f1"]) < 1e-7, (
    "Re-evaluated score differs from saved score; inspect evaluator/state before reporting."
)

print("\nExperiment 3 complete. This is a custom validation result, not an official evaluator score.")


PHASE 2 — ARCH1 + LoRA NER
Device: cuda
GPU: Tesla T4
Arch1 checkpoint: /kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt
Training windows: /kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data/train_windows.pt
Validation windows: /kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data/validation_windows.pt
Output directory: /kaggle/working/phase2_arch1_lora
TRAIN: windows=502, shape=(502, 1024), supervised_tokens=377187
VALIDATION: windows=105, shape=(105, 1024), supervised_tokens=80163
Self-contained data setup: PASS
NER root: /kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER
Tokenizer: /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
Split manifest: /kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data/split_manifest.json
Documents: train= 56 validation= 14
Arch1 strict checkpoint load: PASS | parameters: 88,210,944
Phase-1 c

In [4]:
# ================================================================
# PHASE 2 LEGAL NER — EXPLORATION LAB (NO TRAINING)
# ================================================================
# Purpose:
#   Explore the dataset, labels, token/character boundaries, document
#   split, saved windows, checkpoint metadata, experiment histories and
#   existing validation-error reports BEFORE choosing more experiments.
#
# IMPORTANT:
#   - Does NOT train or update model weights.
#   - Does NOT read test labels (only official test texts for counts/lengths).
#   - Does NOT generate final submissions.
#   - Saves summaries under /kaggle/working/phase2_exploration/.
# ================================================================

import os
import json
import math
import statistics
import traceback
from pathlib import Path
from collections import Counter, defaultdict

import torch
import pandas as pd

try:
    from tokenizers import Tokenizer
except ImportError as exc:
    raise ImportError("Install/enable the `tokenizers` package in Kaggle.") from exc

# ----------------------------------------------------------------
# 0. PATH DISCOVERY
# ----------------------------------------------------------------

NER_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER"),
    Path("/kaggle/working/ner_data/Phase_2_LNER"),
    Path("/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/Phase_2_LNER"),
]

TOKENIZER_CANDIDATES = [
    Path("/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"),
    Path("/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/tokenizer.json"),
    Path("/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokens/tokenizer.json"),
]

DATA_ARTIFACT_ROOTS = [
    Path("/kaggle/working/phase2_arch4_data"),
    Path("/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data"),
]

OUT_DIR = Path("/kaggle/working/phase2_exploration")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def first_existing(paths, required=True):
    for path in paths:
        if path.exists():
            return path
    if required:
        raise FileNotFoundError("None of these paths exists:\n" + "\n".join(map(str, paths)))
    return None

NER_ROOT = first_existing(
    [p for p in NER_ROOT_CANDIDATES if (p / "Train" / "fold1-text.json").is_file()]
)
TOKENIZER_PATH = first_existing(TOKENIZER_CANDIDATES)
DATA_ARTIFACT_ROOT = first_existing(
    [p for p in DATA_ARTIFACT_ROOTS if (p / "split_manifest.json").is_file()],
    required=False,
)
if DATA_ARTIFACT_ROOT is None:
    DATA_ARTIFACT_ROOT = first_existing(DATA_ARTIFACT_ROOTS, required=False)

print("=" * 90)
print("PHASE 2 LEGAL NER — EXPLORATION LAB")
print("No training or model-weight updates will be performed.")
print("=" * 90)
print("NER root:       ", NER_ROOT)
print("Tokenizer:      ", TOKENIZER_PATH)
print("Artifacts root: ", DATA_ARTIFACT_ROOT)
print("Report folder:  ", OUT_DIR)

# ----------------------------------------------------------------
# 1. LOAD SOURCE DATA AND LABELS
# ----------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

fold1_text = load_json(NER_ROOT / "Train" / "fold1-text.json")
fold1_gold = load_json(NER_ROOT / "Train" / "fold1-gold.json")
fold2_text = load_json(NER_ROOT / "Train" / "fold2-text.json")
fold2_gold = load_json(NER_ROOT / "Train" / "fold2-gold.json")
labels_path = NER_ROOT / "Train" / "labels.txt"
with open(labels_path, "r", encoding="utf-8") as f:
    ENTITY_LABELS = [line.strip() for line in f if line.strip()]

TEST_TEXT_PATH = NER_ROOT / "Test" / "fold3-text.json"
test_texts = load_json(TEST_TEXT_PATH)

assert set(fold1_text) == set(fold1_gold), "Fold1 text/gold IDs differ."
assert set(fold2_text) == set(fold2_gold), "Fold2 text/gold IDs differ."
assert not (set(fold1_text) & set(fold2_text)), "Duplicate IDs across training folds."


def text_value(value):
    if isinstance(value, str):
        return value
    if isinstance(value, dict):
        for key in ("text", "content", "document"):
            if isinstance(value.get(key), str):
                return value[key]
    raise TypeError(f"Cannot extract document text from {type(value)}")


def entity_list(value):
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ("entities", "spans", "annotations"):
            if isinstance(value.get(key), list):
                return value[key]
    raise TypeError(f"Cannot extract entity list from {type(value)}")


def parse_entity(ent):
    """Return (start, end, label, optional_surface) from supported annotation shapes."""
    if isinstance(ent, (list, tuple)) and len(ent) >= 3:
        start, end, label = int(ent[0]), int(ent[1]), str(ent[2])
        surface = str(ent[3]) if len(ent) >= 4 and isinstance(ent[3], str) else None
        return start, end, label, surface
    if isinstance(ent, dict):
        start = ent.get("start", ent.get("start_char", ent.get("begin")))
        end = ent.get("end", ent.get("end_char", ent.get("stop")))
        label = ent.get("label", ent.get("type", ent.get("entity_type")))
        surface = ent.get("text", ent.get("surface"))
        if start is not None and end is not None and label is not None:
            return int(start), int(end), str(label), str(surface) if surface is not None else None
    raise ValueError(f"Unrecognized entity annotation: {ent!r}")

all_docs = {}
for fold_name, text_dict, gold_dict in [
    ("fold1", fold1_text, fold1_gold),
    ("fold2", fold2_text, fold2_gold),
]:
    for raw_id, raw_text in text_dict.items():
        doc_id = str(raw_id)
        ents = [parse_entity(e) for e in entity_list(gold_dict[raw_id])]
        all_docs[doc_id] = {
            "text": text_value(raw_text),
            "entities": ents,
            "source_fold": fold_name,
        }

test_docs = {str(k): text_value(v) for k, v in test_texts.items()}
assert len(all_docs) == 70, f"Expected 70 labelled docs, got {len(all_docs)}"
assert len(test_docs) == 35, f"Expected 35 official test docs, got {len(test_docs)}"
assert set(all_docs).isdisjoint(test_docs), "Training/test IDs overlap."

unknown_labels = sorted({e[2] for d in all_docs.values() for e in d["entities"]} - set(ENTITY_LABELS))
assert not unknown_labels, f"Found labels absent from labels.txt: {unknown_labels}"

# ----------------------------------------------------------------
# 2. LOAD SPLIT MANIFEST AND WINDOWS (IF PRESENT)
# ----------------------------------------------------------------

manifest_path = first_existing(
    [root / "split_manifest.json" for root in DATA_ARTIFACT_ROOTS],
    required=False,
)
manifest = load_json(manifest_path) if manifest_path else {}


def manifest_ids(*keys):
    for key in keys:
        value = manifest.get(key) if isinstance(manifest, dict) else None
        if isinstance(value, list):
            return set(map(str, value))
    return set()

TRAIN_IDS = manifest_ids("train_doc_ids", "train_ids", "TRAIN_DOC_IDS")
VAL_IDS = manifest_ids("validation_doc_ids", "val_doc_ids", "validation_ids", "VAL_DOC_IDS")
TEST_IDS = manifest_ids("test_doc_ids", "test_ids", "TEST_DOC_IDS")

if TRAIN_IDS and VAL_IDS:
    assert TRAIN_IDS.isdisjoint(VAL_IDS), "Split manifest has train/validation overlap."
    assert TRAIN_IDS | VAL_IDS == set(all_docs), (
        "Manifest IDs do not exactly cover all 70 labelled docs. "
        f"missing={len(set(all_docs) - (TRAIN_IDS | VAL_IDS))}, "
        f"unknown={len((TRAIN_IDS | VAL_IDS) - set(all_docs))}"
    )
else:
    print("WARNING: Could not find train/validation document IDs in split_manifest.json.")

if TEST_IDS:
    assert TEST_IDS.isdisjoint(TRAIN_IDS | VAL_IDS), "Test IDs overlap train/validation."


def load_window_artifact(filename):
    path = first_existing([root / filename for root in DATA_ARTIFACT_ROOTS], required=False)
    if path is None:
        return None, None
    try:
        obj = torch.load(path, map_location="cpu", weights_only=False)
        return obj, path
    except Exception as exc:
        print(f"WARNING: Could not load {path}: {exc}")
        return None, path

train_windows, train_windows_path = load_window_artifact("train_windows.pt")
val_windows, val_windows_path = load_window_artifact("validation_windows.pt")

# ----------------------------------------------------------------
# 3. TOKEN/OFFSET AUDIT
# ----------------------------------------------------------------

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))


def correct_offsets(text, raw_offsets):
    corrected = []
    for start, end in raw_offsets:
        start, end = int(start), int(end)
        if not (0 <= start <= end <= len(text)):
            raise ValueError(f"Invalid tokenizer offset {(start, end)} for text length {len(text)}")
        while start < end and text[start].isspace():
            start += 1
        corrected.append((start, end))
    return corrected

from bisect import bisect_left, bisect_right

label_counts_total = Counter()
label_counts_train = Counter()
label_counts_val = Counter()
label_doc_presence = Counter()
label_boundary_failures = Counter()
label_uncovered = Counter()
label_token_conflicts = Counter()
entity_char_lengths = defaultdict(list)
entity_token_lengths = defaultdict(list)
doc_rows = []
entity_rows = []
examples_boundary_failures = []
examples_bad_annotations = []
all_token_count = 0
all_char_count = 0
all_entities = 0
bad_offsets_count = 0
bad_surface_count = 0
no_token_overlap_count = 0
boundary_exact_count = 0
entities_with_conflict_count = 0

for doc_id, item in all_docs.items():
    text = item["text"]
    ents = item["entities"]
    enc = tokenizer.encode(text)
    offsets = correct_offsets(text, enc.offsets)
    token_starts = [x[0] for x in offsets]
    n_tokens = len(offsets)
    all_token_count += n_tokens
    all_char_count += len(text)
    per_doc_labels = Counter()
    valid_entities = 0
    doc_conflict_entity_ids = set()
    token_owners = defaultdict(set)

    for ent_idx, (start, end, label, surface) in enumerate(ents):
        all_entities += 1
        label_counts_total[label] += 1
        per_doc_labels[label] += 1
        label_doc_presence[label] += 0  # ensures label key exists
        entity_char_lengths[label].append(end - start)

        if start < 0 or end <= start or end > len(text):
            bad_offsets_count += 1
            if len(examples_bad_annotations) < 20:
                examples_bad_annotations.append({"doc_id": doc_id, "start": start, "end": end, "label": label, "text_len": len(text)})
            continue
        valid_entities += 1
        if surface is not None and surface != text[start:end]:
            bad_surface_count += 1
            if len(examples_bad_annotations) < 20:
                examples_bad_annotations.append({"doc_id": doc_id, "start": start, "end": end, "label": label, "surface_mismatch": surface, "actual": text[start:end]})

        # Token offsets whose starts lie in [start, end), including one prior token
        # which might start before the entity and overlap it.
        lo = max(0, bisect_left(token_starts, start) - 1)
        hi = bisect_left(token_starts, end)
        overlapping = [i for i in range(lo, hi) if offsets[i][0] < offsets[i][1] and offsets[i][1] > start and offsets[i][0] < end]
        if not overlapping:
            no_token_overlap_count += 1
            label_uncovered[label] += 1
            continue

        first, last = overlapping[0], overlapping[-1]
        exact_boundary = offsets[first][0] == start and offsets[last][1] == end
        if exact_boundary:
            boundary_exact_count += 1
        else:
            label_boundary_failures[label] += 1
            if len(examples_boundary_failures) < 40:
                examples_boundary_failures.append({
                    "doc_id": doc_id, "start": start, "end": end, "label": label,
                    "gold_text": text[start:end],
                    "first_token_offset": list(offsets[first]),
                    "last_token_offset": list(offsets[last]),
                    "prefix": text[max(0, start-25):start],
                    "suffix": text[end:min(len(text), end+25)],
                })

        entity_token_lengths[label].append(len(overlapping))
        for ti in overlapping:
            token_owners[ti].add(ent_idx)

        entity_rows.append({
            "doc_id": doc_id,
            "split": "train" if doc_id in TRAIN_IDS else "validation" if doc_id in VAL_IDS else "unknown",
            "label": label,
            "start": start,
            "end": end,
            "char_length": end - start,
            "token_length": len(overlapping),
            "token_boundary_exact": exact_boundary,
            "text": text[start:end],
        })

    for owners in token_owners.values():
        if len(owners) > 1:
            doc_conflict_entity_ids.update(owners)
    entities_with_conflict_count += len(doc_conflict_entity_ids)
    for ent_idx in doc_conflict_entity_ids:
        label_token_conflicts[ents[ent_idx][2]] += 1

    for label in per_doc_labels:
        label_doc_presence[label] += 1

    split_name = "train" if doc_id in TRAIN_IDS else "validation" if doc_id in VAL_IDS else "unknown"
    doc_rows.append({
        "doc_id": doc_id,
        "source_fold": item["source_fold"],
        "split": split_name,
        "characters": len(text),
        "tokens": n_tokens,
        "entities": len(ents),
        "entities_per_1000_tokens": 1000 * len(ents) / max(n_tokens, 1),
        "labels_present": len(per_doc_labels),
        "has_empty_entities": len(ents) == 0,
    })

    if doc_id in TRAIN_IDS:
        label_counts_train.update(per_doc_labels)
    elif doc_id in VAL_IDS:
        label_counts_val.update(per_doc_labels)

# Recount split distributions using exact document partitions.
label_counts_train = Counter()
label_counts_val = Counter()
for doc_id, item in all_docs.items():
    target = label_counts_train if doc_id in TRAIN_IDS else label_counts_val if doc_id in VAL_IDS else None
    if target is not None:
        target.update(ent[2] for ent in item["entities"])

# ----------------------------------------------------------------
# 4. DATA QUALITY, IMBALANCE, LENGTH AND WINDOW SUMMARY
# ----------------------------------------------------------------

print("\n" + "=" * 90)
print("DATASET AND SPLIT")
print("=" * 90)
print(f"Labelled docs: {len(all_docs)} | fold1: {len(fold1_text)} | fold2: {len(fold2_text)} | official test texts: {len(test_docs)}")
print(f"Split manifest: {manifest_path if manifest_path else 'NOT FOUND'}")
print(f"Train docs: {len(TRAIN_IDS) if TRAIN_IDS else 'unknown'} | validation docs: {len(VAL_IDS) if VAL_IDS else 'unknown'}")
print(f"Total gold entities: {all_entities:,} | total tokenizer tokens across labelled docs: {all_token_count:,}")
print(f"Average document length: {statistics.mean([r['tokens'] for r in doc_rows]):.1f} tokens")
print(f"Median document length: {statistics.median([r['tokens'] for r in doc_rows]):.1f} tokens")
print(f"Longest document: {max(r['tokens'] for r in doc_rows)} tokens")
print(f"Character length: mean={statistics.mean([r['characters'] for r in doc_rows]):.1f}, median={statistics.median([r['characters'] for r in doc_rows]):.1f}")
print(f"Invalid entity offsets: {bad_offsets_count} | annotation surface mismatches: {bad_surface_count}")
print(f"Entities with no overlapping tokenizer token: {no_token_overlap_count}")
print(f"Exact token-boundary-aligned entities: {boundary_exact_count}/{all_entities-no_token_overlap_count-bad_offsets_count} "
      f"({100*boundary_exact_count/max(all_entities-no_token_overlap_count-bad_offsets_count,1):.2f}%)")
print(f"Entities with non-exact token boundaries: {sum(label_boundary_failures.values())}")
print(f"Entities involved in overlapping-token annotation conflicts: {entities_with_conflict_count}")

label_rows = []
for label in ENTITY_LABELS:
    train_n = label_counts_train[label]
    val_n = label_counts_val[label]
    total_n = label_counts_total[label]
    token_lengths = entity_token_lengths[label]
    char_lengths = entity_char_lengths[label]
    label_rows.append({
        "label": label,
        "train_entities": train_n,
        "validation_entities": val_n,
        "total_entities": total_n,
        "share_pct": 100 * total_n / max(all_entities, 1),
        "train_docs_with_label": sum(1 for d in all_docs if d in TRAIN_IDS and any(e[2] == label for e in all_docs[d]["entities"])),
        "validation_docs_with_label": sum(1 for d in all_docs if d in VAL_IDS and any(e[2] == label for e in all_docs[d]["entities"])),
        "median_char_length": statistics.median(char_lengths) if char_lengths else 0,
        "median_token_length": statistics.median(token_lengths) if token_lengths else 0,
        "boundary_failures": label_boundary_failures[label],
        "no_token_overlap": label_uncovered[label],
        "token_conflict_entities": label_token_conflicts[label],
    })
label_df = pd.DataFrame(label_rows)
label_df["train_val_ratio"] = label_df.apply(lambda r: r["train_entities"] / max(r["validation_entities"], 1), axis=1)
label_df = label_df.sort_values("total_entities", ascending=True).reset_index(drop=True)

print("\nLABEL IMBALANCE — rarest first")
print(label_df[["label", "train_entities", "validation_entities", "total_entities", "share_pct", "median_char_length", "median_token_length", "boundary_failures"]].to_string(index=False, formatters={"share_pct": "{:.2f}".format, "median_char_length": "{:.1f}".format, "median_token_length": "{:.1f}".format}))

rare_labels = label_df[label_df["total_entities"] < max(200, int(all_entities * 0.02))]["label"].tolist()
max_label_count = int(label_df["total_entities"].max()) if not label_df.empty else 0
min_label_count = int(label_df["total_entities"].min()) if not label_df.empty else 0
imbalance_ratio = max_label_count / max(min_label_count, 1)

print(f"\nLargest/smallest entity-count ratio: {imbalance_ratio:.2f}x")
print("Labels with <200 entities or <2% of all entities:", rare_labels)

# Character and token-length summaries per entity type.
length_rows = []
for label in ENTITY_LABELS:
    chars = entity_char_lengths[label]
    toks = entity_token_lengths[label]
    if not chars:
        continue
    sorted_chars = sorted(chars)
    sorted_toks = sorted(toks)
    p95_index_c = min(len(sorted_chars)-1, int(0.95*(len(sorted_chars)-1)))
    p95_index_t = min(len(sorted_toks)-1, int(0.95*(len(sorted_toks)-1)))
    length_rows.append({
        "label": label,
        "count": len(chars),
        "char_mean": statistics.mean(chars),
        "char_median": statistics.median(chars),
        "char_p95": sorted_chars[p95_index_c],
        "token_mean": statistics.mean(toks) if toks else 0,
        "token_median": statistics.median(toks) if toks else 0,
        "token_p95": sorted_toks[p95_index_t] if toks else 0,
    })
length_df = pd.DataFrame(length_rows)

# Window artifact properties and distributions.
window_rows = []
for name, obj, path in [
    ("train", train_windows, train_windows_path),
    ("validation", val_windows, val_windows_path),
]:
    if obj is None:
        continue
    x = obj.get("input_ids") if isinstance(obj, dict) else None
    y = obj.get("labels") if isinstance(obj, dict) else None
    meta = obj.get("metadata", []) if isinstance(obj, dict) else []
    row = {
        "partition": name,
        "path": str(path),
        "windows": int(x.shape[0]) if torch.is_tensor(x) and x.ndim >= 2 else None,
        "sequence_length": int(x.shape[1]) if torch.is_tensor(x) and x.ndim >= 2 else None,
        "label_shape_matches_input": bool(torch.is_tensor(x) and torch.is_tensor(y) and tuple(x.shape) == tuple(y.shape)),
        "supervised_tokens": int((y != -100).sum().item()) if torch.is_tensor(y) else None,
        "ignored_positions": int((y == -100).sum().item()) if torch.is_tensor(y) else None,
        "metadata_rows": len(meta) if isinstance(meta, (list, tuple)) else None,
        "num_documents_metadata": len({str(m.get("doc_id")) for m in meta if isinstance(m, dict) and "doc_id" in m}),
    }
    if torch.is_tensor(x):
        row["min_token_id"] = int(x.min().item())
        row["max_token_id"] = int(x.max().item())
    if torch.is_tensor(y):
        valid_labels = y[y != -100]
        row["min_supervised_label_id"] = int(valid_labels.min().item()) if valid_labels.numel() else None
        row["max_supervised_label_id"] = int(valid_labels.max().item()) if valid_labels.numel() else None
    window_rows.append(row)

window_df = pd.DataFrame(window_rows)
print("\nWINDOW ARTIFACTS")
if not window_df.empty:
    print(window_df.to_string(index=False))
else:
    print("Saved window artifacts not found; dataset source analysis is still complete.")

# ----------------------------------------------------------------
# 5. EXISTING EXPERIMENT CHECKPOINTS/HISTORIES/ERROR REPORTS
# ----------------------------------------------------------------

ARTIFACT_CANDIDATES = {
    "arch4_full_ft_checkpoint": [
        Path("/kaggle/working/phase2_arch4_baseline/best_strict_macro_f1.pt"),
        Path("/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_baseline/best_strict_macro_f1.pt"),
    ],
    "arch4_lorma_checkpoint": [
        Path("/kaggle/working/phase2_arch4_lorma/best_lorma_strict_macro_f1.pt"),
        Path("/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_lorma/best_lorma_strict_macro_f1.pt"),
    ],
    "arch1_lora_checkpoint": [
        Path("/kaggle/working/phase2_arch1_lora/best_arch1_lora_strict_macro_f1.pt"),
    ],
    "arch4_error_analysis": [
        Path("/kaggle/working/phase2_arch4_baseline/validation_error_analysis.json"),
        Path("/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_baseline/validation_error_analysis.json"),
    ],
    "arch4_lorma_history": [
        Path("/kaggle/working/phase2_arch4_lorma/training_history.json"),
        Path("/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_lorma/training_history.json"),
    ],
    "arch1_lora_history": [
        Path("/kaggle/working/phase2_arch1_lora/training_history.json"),
    ],
}

artifact_paths = {k: first_existing(v, required=False) for k, v in ARTIFACT_CANDIDATES.items()}
experiment_rows = []
checkpoint_details = {}
per_label_experiment_rows = []

for name in ("arch4_full_ft_checkpoint", "arch4_lorma_checkpoint", "arch1_lora_checkpoint"):
    path = artifact_paths[name]
    if path is None:
        experiment_rows.append({"experiment": name, "found": False, "path": None})
        continue
    try:
        # Checkpoints include model weights; mmap lowers transient RAM pressure when supported.
        try:
            ck = torch.load(path, map_location="cpu", weights_only=False, mmap=True)
        except Exception:
            # Compatibility fallback for checkpoints/PyTorch builds that do not support mmap.
            ck = torch.load(path, map_location="cpu", weights_only=False)
        metrics = ck.get("metrics", {}) if isinstance(ck, dict) else {}
        cfg = ck.get("config", ck.get("cfg", {})) if isinstance(ck, dict) else {}
        model_weights_key = next((k for k in ("backbone", "model", "state_dict") if isinstance(ck, dict) and isinstance(ck.get(k), dict)), None)
        row = {
            "experiment": name,
            "found": True,
            "path": str(path),
            "epoch": ck.get("epoch") if isinstance(ck, dict) else None,
            "step": ck.get("step") if isinstance(ck, dict) else None,
            "best_phase1_loss": ck.get("best", ck.get("best_val")) if isinstance(ck, dict) else None,
            "strict_macro_precision": metrics.get("strict_macro_precision"),
            "strict_macro_recall": metrics.get("strict_macro_recall"),
            "strict_macro_f1": metrics.get("strict_macro_f1"),
            "trainable_parameters": cfg.get("trainable_parameters") if isinstance(cfg, dict) else None,
            "file_size_mb": round(path.stat().st_size / (1024**2), 2),
            "model_state_key": model_weights_key,
        }
        experiment_rows.append(row)
        checkpoint_details[name] = {"path": str(path), "metrics": metrics, "epoch": row["epoch"], "step": row["step"]}
        per_label = metrics.get("per_label", {}) if isinstance(metrics, dict) else {}
        for label, lm in per_label.items():
            if isinstance(lm, dict):
                per_label_experiment_rows.append({
                    "experiment": name,
                    "checkpoint": str(path),
                    "label": label,
                    "gold": lm.get("gold"),
                    "predicted": lm.get("predicted"),
                    "tp": lm.get("tp"),
                    "fp": lm.get("fp"),
                    "fn": lm.get("fn"),
                    "precision": lm.get("precision"),
                    "recall": lm.get("recall"),
                    "f1": lm.get("f1"),
                })
        del ck
    except Exception as exc:
        experiment_rows.append({"experiment": name, "found": True, "path": str(path), "inspection_error": repr(exc)})

# Histories can be list of dicts or dict with history/epochs keys.
def load_history_summary(path, experiment):
    if path is None:
        return {"experiment": experiment, "found": False}
    try:
        data = load_json(path)
        if isinstance(data, dict):
            records = data.get("history", data.get("epochs", data.get("records", [])))
            if not isinstance(records, list):
                records = []
        elif isinstance(data, list):
            records = data
        else:
            records = []
        f1key = next((k for k in ("strict_macro_f1", "macro_f1", "val_f1") if any(isinstance(r, dict) and k in r for r in records)), None)
        best_record = max((r for r in records if isinstance(r, dict) and f1key and isinstance(r.get(f1key), (int, float))), key=lambda r: r[f1key], default=None)
        return {
            "experiment": experiment,
            "found": True,
            "path": str(path),
            "num_records": len(records),
            "f1_key": f1key,
            "best_epoch": best_record.get("epoch") if best_record else None,
            "best_f1_from_history": best_record.get(f1key) if best_record else None,
            "history_keys": sorted({k for r in records if isinstance(r, dict) for k in r.keys()}),
        }
    except Exception as exc:
        return {"experiment": experiment, "found": True, "path": str(path), "error": repr(exc)}

history_rows = [
    load_history_summary(artifact_paths["arch4_lorma_history"], "arch4_lorma_history"),
    load_history_summary(artifact_paths["arch1_lora_history"], "arch1_lora_history"),
]

error_summary = {}
error_path = artifact_paths["arch4_error_analysis"]
if error_path:
    try:
        err = load_json(error_path)
        fp = err.get("false_positives", []) if isinstance(err, dict) else []
        fn = err.get("false_negatives", []) if isinstance(err, dict) else []
        by_label = defaultdict(Counter)
        for item in fp:
            if isinstance(item, dict):
                by_label[str(item.get("label", "UNKNOWN"))]["FP"] += 1
        for item in fn:
            if isinstance(item, dict):
                by_label[str(item.get("label", "UNKNOWN"))]["FN"] += 1
        error_summary = {
            "path": str(error_path),
            "gold_entities": err.get("gold_entities"),
            "predicted_entities": err.get("predicted_entities"),
            "exact_matches": err.get("exact_matches"),
            "false_positive_count_reported": err.get("false_positive_count"),
            "false_negative_count_reported": err.get("false_negative_count"),
            "false_positive_records": len(fp),
            "false_negative_records": len(fn),
            "per_label_fp_fn": {k: dict(v) for k, v in by_label.items()},
        }
    except Exception as exc:
        error_summary = {"path": str(error_path), "error": repr(exc)}

print("\n" + "=" * 90)
print("EXISTING EXPERIMENT ARTIFACTS")
print("=" * 90)
experiment_df = pd.DataFrame(experiment_rows)
print(experiment_df.to_string(index=False) if not experiment_df.empty else "No experiment checkpoints found.")
print("\nHistory summaries:")
print(pd.DataFrame(history_rows).to_string(index=False) if history_rows else "No histories found.")
print("\nPer-label metrics found inside saved checkpoints:")
per_label_df = pd.DataFrame(per_label_experiment_rows)
if not per_label_df.empty:
    print(per_label_df[["experiment", "label", "gold", "predicted", "tp", "fp", "fn", "precision", "recall", "f1"]].to_string(index=False))
else:
    print("No saved per-label metrics found in checkpoint metadata.")
print("\nArch4 error-analysis summary:")
print(json.dumps(error_summary, indent=2)[:12000] if error_summary else "No saved error-analysis JSON found.")

# ----------------------------------------------------------------
# 6. DECISION-ORIENTED RECOMMENDATIONS
# ----------------------------------------------------------------

recommendations = []
if bad_offsets_count or bad_surface_count:
    recommendations.append("DATA QUALITY: inspect invalid gold offsets/surface mismatches before any modelling changes.")
if sum(label_boundary_failures.values()) > 0:
    recommendations.append(
        f"BOUNDARIES: {sum(label_boundary_failures.values())} entities are not exactly aligned to tokenizer boundaries. "
        "Character-span diagnostics are useful; do not blindly trim punctuation."
    )
if rare_labels:
    recommendations.append("IMBALANCE: consider a controlled class-weighted-loss experiment; calculate weights from TRAIN labels only and cap extremes.")
if error_summary.get("per_label_fp_fn"):
    recommendations.append("ERROR ANALYSIS: identify labels with high FP/FN counts and inspect representative saved examples before selecting a remedy.")
recommendations.append("COMPARABILITY: keep the same document split and strict span evaluator across all experiments; report custom scores until checked against the official evaluator.")
recommendations.append("CONTROL: Arch1 Full Fine-Tuning would isolate architecture vs LoRA; alternatively keep Arch1 LoRA as the hard-label control for Arch1+LoRA distillation.")
recommendations.append("METHOD REQUIREMENTS: retain Full Fine-Tuning and LoRMA+; complete genuinely distinct BitFit, Houlsby Adapter, LoRA and knowledge-distillation runs as required.")
recommendations.append("ARCH2: rebuild windows at its 768-token context. Do not feed the 1024-token Arch4 windows directly to Arch2.")
recommendations.append("FINAL DATA: keep Fold 3 test text separate during model selection. Decide on all-70-document retraining only after comparing validation results.")

print("\n" + "=" * 90)
print("DECISION NOTES — READ BEFORE CHOOSING THE NEXT IMPROVEMENT")
print("=" * 90)
for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

# ----------------------------------------------------------------
# 7. SAVE REPORTS
# ----------------------------------------------------------------

pd.DataFrame(doc_rows).to_csv(OUT_DIR / "document_overview.csv", index=False)
label_df.to_csv(OUT_DIR / "label_distribution_and_boundaries.csv", index=False)
length_df.to_csv(OUT_DIR / "entity_length_summary.csv", index=False)
if not window_df.empty:
    window_df.to_csv(OUT_DIR / "window_artifact_summary.csv", index=False)
experiment_df.to_csv(OUT_DIR / "checkpoint_summary.csv", index=False)
pd.DataFrame(history_rows).to_csv(OUT_DIR / "history_summary.csv", index=False)
per_label_df.to_csv(OUT_DIR / "per_label_experiment_comparison.csv", index=False)

with open(OUT_DIR / "boundary_failure_examples.json", "w", encoding="utf-8") as f:
    json.dump(examples_boundary_failures, f, indent=2, ensure_ascii=False)
with open(OUT_DIR / "annotation_quality_examples.json", "w", encoding="utf-8") as f:
    json.dump(examples_bad_annotations, f, indent=2, ensure_ascii=False)
with open(OUT_DIR / "arch4_error_summary.json", "w", encoding="utf-8") as f:
    json.dump(error_summary, f, indent=2, ensure_ascii=False)

report = {
    "purpose": "Phase 2 Legal NER exploratory audit; no training performed",
    "ner_root": str(NER_ROOT),
    "tokenizer_path": str(TOKENIZER_PATH),
    "split_manifest": str(manifest_path) if manifest_path else None,
    "labelled_documents": len(all_docs),
    "train_documents_from_manifest": len(TRAIN_IDS),
    "validation_documents_from_manifest": len(VAL_IDS),
    "official_test_documents_text_only": len(test_docs),
    "entity_labels": ENTITY_LABELS,
    "total_entities": all_entities,
    "total_tokens_labelled_docs": all_token_count,
    "invalid_offsets": bad_offsets_count,
    "surface_mismatches": bad_surface_count,
    "no_token_overlap": no_token_overlap_count,
    "token_boundary_exact_entities": boundary_exact_count,
    "token_boundary_failures": sum(label_boundary_failures.values()),
    "token_conflict_entities": entities_with_conflict_count,
    "imbalance_ratio_max_over_min_label_count": imbalance_ratio,
    "rare_labels_threshold": max(200, int(all_entities * 0.02)),
    "rare_labels": rare_labels,
    "window_artifacts": window_rows,
    "experiment_checkpoints": experiment_rows,
    "per_label_experiment_metrics": per_label_experiment_rows,
    "history_summaries": history_rows,
    "arch4_error_summary": error_summary,
    "recommendations": recommendations,
    "note": "Phase-1 model validation losses are not Phase-2 NER scores. Existing custom strict-span metrics are not official until cross-checked.",
}
with open(OUT_DIR / "exploration_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

markdown = [
    "# Phase 2 Legal NER Exploration Report",
    "",
    "This report is an exploratory data/artifact audit. It performed no training and did not read test labels.",
    "",
    f"- Labelled documents: {len(all_docs)}",
    f"- Training documents in manifest: {len(TRAIN_IDS)}",
    f"- Validation documents in manifest: {len(VAL_IDS)}",
    f"- Official test documents (text only): {len(test_docs)}",
    f"- Entity labels: {len(ENTITY_LABELS)}",
    f"- Gold entities: {all_entities}",
    f"- Token boundary failures: {sum(label_boundary_failures.values())}",
    f"- Invalid offsets: {bad_offsets_count}",
    f"- Annotation surface mismatches: {bad_surface_count}",
    "",
    "## Recommended next decisions",
    *[f"{i}. {r}" for i, r in enumerate(recommendations, 1)],
    "",
    "## Saved artifacts",
    *[f"- `{p.name}`" for p in sorted(OUT_DIR.iterdir()) if p.is_file() and p.name != "exploration_report.md"],
    "",
    "Caution: custom evaluator metrics must not be described as official scores until cross-checked with the course evaluator.",
]
(OUT_DIR / "exploration_report.md").write_text("\n".join(markdown), encoding="utf-8")

print("\n" + "=" * 90)
print("EXPLORATION COMPLETE")
print("=" * 90)
print("No model training was performed.")
print("Reports saved to:", OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.name:42s} {p.stat().st_size/1024:8.1f} KB")


PHASE 2 LEGAL NER — EXPLORATION LAB
No training or model-weight updates will be performed.
NER root:        /kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER
Tokenizer:       /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
Artifacts root:  /kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data
Report folder:   /kaggle/working/phase2_exploration

DATASET AND SPLIT
Labelled docs: 70 | fold1: 35 | fold2: 35 | official test texts: 35
Split manifest: /kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data/split_manifest.json
Train docs: 56 | validation docs: 14
Total gold entities: 8,393 | total tokenizer tokens across labelled docs: 457,384
Average document length: 6534.1 tokens
Median document length: 5487.5 tokens
Longest document: 23623 tokens
Character length: mean=31623.5, median=26620.0
Invalid entity offsets: 0 | annotation surface mismatches: 0
Entities with no overlapping tokenizer token:

In [5]:
!python /kaggle/working/diagnose_arch1_lora.py

python3: can't open file '/kaggle/working/diagnose_arch1_lora.py': [Errno 2] No such file or directory


In [6]:
#!/usr/bin/env python3
"""Validation-only diagnostic for the Phase-2 Arch1 + LoRA NER checkpoint.

Run from a Kaggle notebook with the Phase-2 dataset mounted. This script:
  1) restores the saved Arch1+LoRA model and BIO head strictly;
  2) audits checkpoint BIO mapping, manifest/window labels, token coverage,
     token-level confusion, BIO transitions, and character offsets;
  3) re-evaluates document-level exact character spans on validation only;
  4) saves per-label metrics and representative FP/FN examples;
  5) recommends mapping/decoder repair versus a model retrain based on checks.

It does not train anything and never reads the official Fold 3 test text.
"""
from __future__ import annotations

import json
import math
import os
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any

import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------ Paths ---------------------------------------
CHECKPOINT_CANDIDATES = [
    Path("/kaggle/working/phase2_arch1_lora/best_arch1_lora_strict_macro_f1.pt"),
    Path("/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch1_lora/best_arch1_lora_strict_macro_f1.pt"),
]
DATA_DIR_CANDIDATES = [
    Path("/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data"),
    Path("/kaggle/working/phase2_arch4_data"),
]
NER_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER"),
    Path("/kaggle/working/ner_data/Phase_2_LNER"),
    Path("/kaggle/input/ner_data/Phase_2_LNER"),
]
TOKENIZER_CANDIDATES = [
    Path("/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"),
    Path("/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/tokenizer.json"),
    Path("/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokens/tokenizer.json"),
]
OUT_DIR = Path("/kaggle/working/phase2_arch1_lora_diagnostic")
MAX_LEN = 1024
VOCAB_SIZE = 40000
NUM_CLASSES = 25
IGNORE_INDEX = -100

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"


def first_existing(paths: list[Path], description: str) -> Path:
    for path in paths:
        exists = path.is_file() if path.suffix else path.is_dir()
        if exists:
            return path
    raise FileNotFoundError(
        f"Could not find {description}. Checked:\n" + "\n".join(map(str, paths))
    )


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def normalize_text_map(raw: dict, name: str) -> dict[str, str]:
    out = {}
    for key, value in raw.items():
        if isinstance(value, dict):
            value = value.get("text")
        if not isinstance(value, str):
            raise TypeError(f"{name}[{key!r}] is not text")
        out[str(key)] = value
    return out


def normalize_gold_map(raw: dict, name: str) -> dict[str, list]:
    out = {str(key): value for key, value in raw.items()}
    for key, entities in out.items():
        if not isinstance(entities, list):
            raise TypeError(f"{name}[{key!r}] must be a list")
    return out


def read_entity(entity: Any) -> tuple[int, int, str]:
    """Support the dataset's [start, end, label] form and common dict variants."""
    if isinstance(entity, dict):
        start = entity.get("start", entity.get("start_char", entity.get("begin")))
        end = entity.get("end", entity.get("end_char", entity.get("stop")))
        label = entity.get("label", entity.get("type", entity.get("entity_type")))
        if start is None or end is None or label is None:
            raise ValueError(f"Unrecognized entity dictionary: {entity}")
        return int(start), int(end), str(label)
    if not isinstance(entity, (list, tuple)) or len(entity) < 3:
        raise ValueError(f"Unrecognized entity item: {entity!r}")
    return int(entity[0]), int(entity[1]), str(entity[2])


def correct_offsets(text: str, raw_offsets: list[tuple[int, int]]) -> list[tuple[int, int]]:
    """Exact same offset correction used during Arch1+LoRA training/evaluation."""
    corrected = []
    for start, end in raw_offsets:
        start, end = int(start), int(end)
        if not (0 <= start <= end <= len(text)):
            raise ValueError(f"Invalid offset {(start, end)} for text length {len(text)}")
        while start < end and text[start].isspace():
            start += 1
        corrected.append((start, end))
    return corrected


# --------------------------- Arch1 + LoRA model -----------------------------
class Arch1CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, block_size: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("mask", mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq_len, channels = x.shape
        q = self.q_proj(x).view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.mask[:, :, :seq_len, :seq_len] == 0, float("-inf"))
        attn = F.softmax(scores, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(batch, seq_len, channels)
        return self.out_proj(out)


class Arch1FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(F.gelu(self.fc1(x)))


class Arch1TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, block_size: int):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = Arch1CausalSelfAttention(d_model, n_heads, block_size)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = Arch1FeedForward(d_model, d_ff)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class Arch1LegalDecoderLM(nn.Module):
    def __init__(self, vocab_size=40000, block_size=1024, d_model=768,
                 n_layers=8, n_heads=8, d_ff=3072):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([
            Arch1TransformerBlock(d_model, n_heads, d_ff, block_size)
            for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

    def forward_hidden(self, input_ids: torch.Tensor) -> torch.Tensor:
        batch, seq_len = input_ids.shape
        if seq_len > self.block_size:
            raise ValueError(f"Sequence length {seq_len} exceeds {self.block_size}")
        positions = torch.arange(seq_len, device=input_ids.device)
        x = self.token_emb(input_ids) + self.pos_emb(positions)
        for block in self.blocks:
            x = block(x)
        return self.final_norm(x)


class LoRALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, rank=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base_layer
        for p in self.base.parameters():
            p.requires_grad = False
        self.rank = int(rank)
        self.alpha = float(alpha)
        self.scaling = self.alpha / self.rank
        self.dropout = nn.Dropout(dropout)
        self.lora_A = nn.Linear(base_layer.in_features, self.rank, bias=False)
        self.lora_B = nn.Linear(self.rank, base_layer.out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.base(x) + self.scaling * self.lora_B(self.lora_A(self.dropout(x)))


def inject_lora(model: Arch1LegalDecoderLM) -> int:
    targets = [
        ("attn", "q_proj"), ("attn", "k_proj"),
        ("attn", "v_proj"), ("attn", "out_proj"),
        ("mlp", "fc1"), ("mlp", "fc2"),
    ]
    count = 0
    for block in model.blocks:
        for parent_name, attr in targets:
            parent = getattr(block, parent_name)
            old = getattr(parent, attr)
            setattr(parent, attr, LoRALinear(old, rank=8, alpha=16, dropout=0.05))
            count += 1
    return count


def decode_bio_spans(tag_ids: list[int], offsets: list[tuple[int, int]], id2tag: dict[int, str]):
    """Same decoder behavior as training: a standalone I-X starts a new X span."""
    if len(tag_ids) != len(offsets):
        raise ValueError(f"tag/offset length mismatch: {len(tag_ids)} vs {len(offsets)}")
    spans = []
    readable = ["O" if s >= e else id2tag[int(t)] for t, (s, e) in zip(tag_ids, offsets)]
    i = 0
    while i < len(readable):
        tag = readable[i]
        if tag == "O":
            i += 1
            continue
        if not (tag.startswith("B-") or tag.startswith("I-")):
            i += 1
            continue
        label = tag[2:]
        start_char = offsets[i][0]
        j = i + 1
        while j < len(readable) and readable[j] == f"I-{label}":
            j += 1
        end_char = offsets[j - 1][1]
        if start_char < end_char:
            spans.append((start_char, end_char, label, i, j))
        i = j
    return spans, readable


def macro_metrics(gold: Counter, pred: Counter, labels: list[str]) -> dict:
    matched = gold & pred
    tp_by_label = Counter()
    for span, count in matched.items():
        tp_by_label[span[3]] += count
    per_label = {}
    for label in labels:
        gold_n = sum(n for sp, n in gold.items() if sp[3] == label)
        pred_n = sum(n for sp, n in pred.items() if sp[3] == label)
        tp = tp_by_label[label]
        fp, fn = pred_n - tp, gold_n - tp
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        per_label[label] = {
            "gold": int(gold_n), "predicted": int(pred_n), "tp": int(tp),
            "fp": int(fp), "fn": int(fn), "precision": p, "recall": r, "f1": f,
        }
    return {
        "strict_macro_precision": sum(v["precision"] for v in per_label.values()) / len(labels),
        "strict_macro_recall": sum(v["recall"] for v in per_label.values()) / len(labels),
        "strict_macro_f1": sum(v["f1"] for v in per_label.values()) / len(labels),
        "gold_entities": int(sum(gold.values())),
        "predicted_entities": int(sum(pred.values())),
        "exact_matches": int(sum(matched.values())),
        "per_label": per_label,
    }


def safe_surface(text: str, start: int, end: int, radius: int = 80) -> str:
    left, right = max(0, start - radius), min(len(text), end + radius)
    return text[left:right].replace("\n", " ").replace("\r", " ")


# ------------------------------ Load data -----------------------------------
checkpoint_path = first_existing(CHECKPOINT_CANDIDATES, "Arch1+LoRA NER checkpoint")
data_dir = first_existing(DATA_DIR_CANDIDATES, "Phase-2 cached-window directory")
ner_root = first_existing(NER_ROOT_CANDIDATES, "Phase-2 NER source directory")
tokenizer_path = first_existing(TOKENIZER_CANDIDATES, "Phase-1 tokenizer.json")

train_windows_path = data_dir / "train_windows.pt"
val_windows_path = data_dir / "validation_windows.pt"
manifest_path = data_dir / "split_manifest.json"
for p in [train_windows_path, val_windows_path, manifest_path]:
    if not p.is_file():
        raise FileNotFoundError(f"Missing required file: {p}")

print("=" * 88)
print("ARCH1 + LoRA NER — DIAGNOSTIC (validation split only)")
print("=" * 88)
print("Device:", DEVICE)
if USE_AMP:
    print("GPU:", torch.cuda.get_device_name(0))
print("NER checkpoint:", checkpoint_path)
print("Data directory:", data_dir)
print("NER source:", ner_root)
print("Tokenizer:", tokenizer_path)

from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(tokenizer_path))
with (ner_root / "Train/labels.txt").open("r", encoding="utf-8") as f:
    ENTITY_LABELS = [line.strip() for line in f if line.strip()]
if len(ENTITY_LABELS) != 12:
    raise ValueError(f"Expected 12 entity labels, found {len(ENTITY_LABELS)}: {ENTITY_LABELS}")
EXPECTED_BIO_TAGS = ["O"] + [f"B-{x}" for x in ENTITY_LABELS] + [f"I-{x}" for x in ENTITY_LABELS]
EXPECTED_TAG2ID = {tag: i for i, tag in enumerate(EXPECTED_BIO_TAGS)}
EXPECTED_ID2TAG = {i: tag for tag, i in EXPECTED_TAG2ID.items()}

f1_text = normalize_text_map(load_json(ner_root / "Train/fold1-text.json"), "fold1_text")
f1_gold = normalize_gold_map(load_json(ner_root / "Train/fold1-gold.json"), "fold1_gold")
f2_text = normalize_text_map(load_json(ner_root / "Train/fold2-text.json"), "fold2_text")
f2_gold = normalize_gold_map(load_json(ner_root / "Train/fold2-gold.json"), "fold2_gold")
if set(f1_text) != set(f1_gold) or set(f2_text) != set(f2_gold):
    raise ValueError("Source text IDs and gold IDs do not align within a fold")
if set(f1_text) & set(f2_text):
    raise ValueError("Duplicate document IDs appear across folds")
all_docs: dict[str, dict] = {}
for texts, golds in [(f1_text, f1_gold), (f2_text, f2_gold)]:
    for doc_id, text in texts.items():
        all_docs[doc_id] = {"text": text, "entities": golds[doc_id]}
if len(all_docs) != 70:
    raise ValueError(f"Expected exactly 70 labelled docs, found {len(all_docs)}")

manifest = load_json(manifest_path)
train_ids = [str(x) for x in manifest["train_doc_ids"]]
val_ids = [str(x) for x in manifest["validation_doc_ids"]]
if len(train_ids) != 56 or len(val_ids) != 14:
    raise ValueError(f"Expected 56/14 train/val docs, found {len(train_ids)}/{len(val_ids)}")
if set(train_ids) & set(val_ids) or (set(train_ids) | set(val_ids)) != set(all_docs):
    raise ValueError("Split manifest does not partition the 70 labelled documents exactly")
val_docs = {doc_id: all_docs[doc_id] for doc_id in val_ids}

train_data = torch.load(train_windows_path, map_location="cpu", weights_only=False)
val_data = torch.load(val_windows_path, map_location="cpu", weights_only=False)
for name, data in [("train", train_data), ("validation", val_data)]:
    x, y, metadata = data["input_ids"], data["labels"], data["metadata"]
    assert x.ndim == y.ndim == 2 and x.shape == y.shape
    assert x.shape[1] == MAX_LEN and x.dtype == torch.long and y.dtype == torch.long
    assert len(metadata) == x.shape[0]
    active = y[y != IGNORE_INDEX]
    assert active.numel() > 0 and int(active.min()) >= 0 and int(active.max()) < NUM_CLASSES
    print(f"{name.title()} windows: {len(metadata)} | shape={tuple(x.shape)} | supervised={int(active.numel()):,}")

train_window_ids = {str(m["doc_id"]) for m in train_data["metadata"]}
val_window_ids = {str(m["doc_id"]) for m in val_data["metadata"]}
window_partition_ok = (train_window_ids == set(train_ids) and val_window_ids == set(val_ids)
                       and not (train_window_ids & val_window_ids))
print("Window document split matches manifest:", "PASS" if window_partition_ok else "FAIL")

# --------------------------- Check checkpoint mapping ----------------------
ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
if not isinstance(ckpt, dict) or "model" not in ckpt or "bio_head" not in ckpt:
    raise ValueError("This does not look like the saved Arch1+LoRA NER checkpoint (missing model/bio_head)")
saved_tag2id_raw = ckpt.get("tag2id")
saved_tag2id = None
if isinstance(saved_tag2id_raw, dict):
    saved_tag2id = {str(k): int(v) for k, v in saved_tag2id_raw.items()}
else:
    print("WARNING: checkpoint does not store tag2id; using labels.txt ordering for diagnostic decode.")

mapping_matches_labels = saved_tag2id is not None and saved_tag2id == EXPECTED_TAG2ID
manifest_tags = manifest.get("bio_tags")
manifest_mapping_matches = manifest_tags is None or list(manifest_tags) == EXPECTED_BIO_TAGS
# Decode using the mapping that the model was actually trained with if available.
if saved_tag2id is not None:
    model_id2tag = {idx: tag for tag, idx in saved_tag2id.items()}
    if len(model_id2tag) != NUM_CLASSES or set(model_id2tag) != set(range(NUM_CLASSES)):
        raise ValueError(f"Checkpoint tag2id is not a complete 25-class mapping: {saved_tag2id}")
else:
    model_id2tag = EXPECTED_ID2TAG

print("Labels.txt order:", ENTITY_LABELS)
print("Checkpoint BIO map matches labels.txt:", "PASS" if mapping_matches_labels else "FAIL")
print("Manifest BIO order matches labels.txt:", "PASS" if manifest_mapping_matches else "FAIL")
if saved_tag2id is not None and not mapping_matches_labels:
    print("Expected mapping:", EXPECTED_TAG2ID)
    print("Checkpoint mapping:", saved_tag2id)
if not manifest_mapping_matches:
    print("Manifest BIO tags:", manifest_tags)

# -------------------------- Restore model strictly -------------------------
model = Arch1LegalDecoderLM(vocab_size=VOCAB_SIZE, block_size=MAX_LEN)
inserted = inject_lora(model)
if inserted != 48:
    raise AssertionError(f"Expected 48 LoRA modules, got {inserted}")
missing, unexpected = model.load_state_dict(ckpt["model"], strict=True)
# strict=True raises if mismatched; retain this assertion for readability.
if missing or unexpected:
    raise RuntimeError(f"Checkpoint state mismatch: missing={missing}, unexpected={unexpected}")
head = nn.Linear(768, NUM_CLASSES)
head.load_state_dict(ckpt["bio_head"], strict=True)
model = model.to(DEVICE).eval()
head = head.to(DEVICE).eval()
print("Strict model + BIO-head restore: PASS | LoRA modules:", inserted)
print("Checkpoint epoch:", ckpt.get("epoch"))
print("Stored validation F1:", (ckpt.get("metrics") or {}).get("strict_macro_f1"))

# -------------------------- Run validation inference ------------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)
val_by_id = {str(k): v for k, v in val_docs.items()}
offsets_by_doc: dict[str, list[tuple[int, int]]] = {}
pred_tags_by_doc: dict[str, list[int]] = {}
pred_conf_by_doc: dict[str, list[float]] = {}
ownership_by_doc: dict[str, list[bool]] = {}

for doc_id, item in val_by_id.items():
    enc = tokenizer.encode(item["text"])
    offsets = correct_offsets(item["text"], enc.offsets)
    offsets_by_doc[doc_id] = offsets
    pred_tags_by_doc[doc_id] = [0] * len(offsets)
    pred_conf_by_doc[doc_id] = [0.0] * len(offsets)
    ownership_by_doc[doc_id] = [False] * len(offsets)

# Token-label confusion is calculated from supervised validation positions in cached windows.
token_confusion = Counter()
raw_pred_token_counts = Counter()
raw_gold_token_counts = Counter()
invalid_i_transitions = 0
all_i_transitions = 0
max_conf_by_doc = pred_conf_by_doc

with torch.inference_mode():
    for wi, meta in enumerate(val_data["metadata"]):
        doc_id = str(meta["doc_id"])
        if doc_id not in offsets_by_doc:
            raise KeyError(f"Window references unknown validation doc: {doc_id}")
        x = val_data["input_ids"][wi:wi+1].to(DEVICE)
        y = val_data["labels"][wi].tolist()
        if USE_AMP:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = head(model.forward_hidden(x))
        else:
            logits = head(model.forward_hidden(x))
        logits = logits.float()[0]
        probs, pred_ids = torch.softmax(logits, dim=-1).max(dim=-1)
        pred_ids = pred_ids.cpu().tolist()
        probs = probs.cpu().tolist()

        for pos, gold_id in enumerate(y):
            if gold_id == IGNORE_INDEX:
                continue
            pred_id = int(pred_ids[pos])
            token_confusion[(int(gold_id), pred_id)] += 1
            raw_pred_token_counts[pred_id] += 1
            raw_gold_token_counts[int(gold_id)] += 1

        window_start = int(meta["window_start"])
        valid_length = int(meta["valid_length"])
        owner_local_start = int(meta["owner_local_start"])
        if not (0 <= owner_local_start <= valid_length <= MAX_LEN):
            raise ValueError(f"Bad window bounds in metadata row {wi}: {meta}")
        for local_idx in range(owner_local_start, valid_length):
            global_idx = window_start + local_idx
            if global_idx >= len(offsets_by_doc[doc_id]):
                raise IndexError(f"Token outside document {doc_id}: {global_idx}")
            if ownership_by_doc[doc_id][global_idx]:
                raise AssertionError(f"Duplicate token owner: {doc_id}/{global_idx}")
            pred_tags_by_doc[doc_id][global_idx] = int(pred_ids[local_idx])
            pred_conf_by_doc[doc_id][global_idx] = float(probs[local_idx])
            ownership_by_doc[doc_id][global_idx] = True

coverage_ok = True
coverage_missing = {}
for doc_id, ownership in ownership_by_doc.items():
    missing_n = ownership.count(False)
    if missing_n:
        coverage_ok = False
        coverage_missing[doc_id] = missing_n
print("All validation tokens receive exactly one prediction:", "PASS" if coverage_ok else "FAIL")
if coverage_missing:
    print("Missing ownership counts:", coverage_missing)

# Validate that current model's token IDs can be interpreted as BIO tags.
for doc_id, pred_ids in pred_tags_by_doc.items():
    for tag_id in pred_ids:
        if tag_id not in model_id2tag:
            raise ValueError(f"Predicted class ID {tag_id} not in checkpoint BIO mapping")

# Gold and predicted exact character spans, plus confidence for debugging.
gold_spans = Counter()
pred_spans = Counter()
pred_span_details = []
invalid_i_by_doc = {}
for doc_id, item in val_by_id.items():
    text = item["text"]
    offsets = offsets_by_doc[doc_id]
    tags = pred_tags_by_doc[doc_id]
    if len(offsets) != len(tags):
        raise AssertionError(f"Tokenizer/prediction length mismatch: {doc_id}: {len(offsets)} != {len(tags)}")
    for ent in item["entities"]:
        start, end, label = read_entity(ent)
        if not (0 <= start <= end <= len(text)):
            raise ValueError(f"Gold span out of bounds in {doc_id}: {(start, end, label)}")
        gold_spans[(doc_id, start, end, label)] += 1

    readable_tags = []
    for tag_id, (start, end) in zip(tags, offsets):
        readable_tags.append("O" if start >= end else model_id2tag[int(tag_id)])
    bad_i = 0
    i_tags = 0
    for i, tag in enumerate(readable_tags):
        if tag.startswith("I-"):
            i_tags += 1
            label = tag[2:]
            if i == 0 or readable_tags[i-1] not in (f"B-{label}", f"I-{label}"):
                bad_i += 1
    invalid_i_transitions += bad_i
    all_i_transitions += i_tags
    invalid_i_by_doc[doc_id] = {"invalid_I_transitions": bad_i, "I_tags": i_tags}

    spans, _ = decode_bio_spans(tags, offsets, model_id2tag)
    for start, end, label, token_start, token_end in spans:
        key = (doc_id, start, end, label)
        pred_spans[key] += 1
        confs = pred_conf_by_doc[doc_id][token_start:token_end]
        pred_span_details.append({
            "doc_id": doc_id, "start": start, "end": end, "label": label,
            "confidence_mean": (sum(confs) / len(confs)) if confs else 0.0,
            "confidence_min": min(confs) if confs else 0.0,
            "text": text[start:end],
        })

metrics = macro_metrics(gold_spans, pred_spans, ENTITY_LABELS)
stored_metrics = ckpt.get("metrics") or {}
recomputed_f1 = float(metrics["strict_macro_f1"])
stored_f1 = stored_metrics.get("strict_macro_f1")
f1_reproduces = stored_f1 is not None and abs(recomputed_f1 - float(stored_f1)) <= 1e-4

# Offset/alignment diagnostics on validation gold entities.
exact_boundary_count = 0
nonexact_boundary_count = 0
invalid_gold_surface_count = 0
for doc_id, item in val_by_id.items():
    text = item["text"]
    offsets = offsets_by_doc[doc_id]
    token_starts = {s for s, e in offsets if s < e}
    token_ends = {e for s, e in offsets if s < e}
    for ent in item["entities"]:
        start, end, label = read_entity(ent)
        if not text[start:end]:
            invalid_gold_surface_count += 1
        if start in token_starts and end in token_ends:
            exact_boundary_count += 1
        else:
            nonexact_boundary_count += 1

# Save a list of exact FP/FN examples and close same-label overlaps for diagnosis.
matched = gold_spans & pred_spans
fp_keys = list((pred_spans - gold_spans).elements())
fn_keys = list((gold_spans - pred_spans).elements())

# Build per-label output and examples, with overlapping gold candidates to reveal boundary-only misses.
def nearest_gold_same_label(doc_id: str, start: int, end: int, label: str):
    cands = []
    for gdoc, gs, ge, glabel in gold_spans:
        if gdoc != doc_id or glabel != label:
            continue
        overlap = max(0, min(end, ge) - max(start, gs))
        distance = max(0, max(gs - end, start - ge))
        if overlap > 0 or distance <= 20:
            cands.append((overlap, -distance, gs, ge, glabel))
    cands.sort(reverse=True)
    if not cands:
        return None
    _, _, gs, ge, glabel = cands[0]
    return {"start": gs, "end": ge, "label": glabel, "text": val_by_id[doc_id]["text"][gs:ge]}

fp_records = []
for key in fp_keys:
    doc_id, start, end, label = key
    text = val_by_id[doc_id]["text"]
    matching_detail = next((d for d in pred_span_details if (d["doc_id"], d["start"], d["end"], d["label"]) == key), {})
    fp_records.append({
        "kind": "FP", "doc_id": doc_id, "start": start, "end": end, "label": label,
        "text": text[start:end], "context": safe_surface(text, start, end),
        "nearest_gold_same_label": nearest_gold_same_label(doc_id, start, end, label),
        "confidence_mean": matching_detail.get("confidence_mean"),
    })
fn_records = []
for key in fn_keys:
    doc_id, start, end, label = key
    text = val_by_id[doc_id]["text"]
    # Find best overlapping/well-near prediction of same label.
    cands = []
    for pdoc, ps, pe, plabel in pred_spans:
        if pdoc != doc_id or plabel != label:
            continue
        overlap = max(0, min(end, pe) - max(start, ps))
        distance = max(0, max(ps - end, start - pe))
        if overlap > 0 or distance <= 20:
            cands.append((overlap, -distance, ps, pe, plabel))
    cands.sort(reverse=True)
    nearest_pred = None
    if cands:
        _, _, ps, pe, plabel = cands[0]
        nearest_pred = {"start": ps, "end": pe, "label": plabel, "text": text[ps:pe]}
    fn_records.append({
        "kind": "FN", "doc_id": doc_id, "start": start, "end": end, "label": label,
        "text": text[start:end], "context": safe_surface(text, start, end),
        "nearest_prediction_same_label": nearest_pred,
    })

# Stable ordering: examples with nearest boundary overlap first, then label.
fp_records.sort(key=lambda x: (x["nearest_gold_same_label"] is not None,
                               x["confidence_mean"] if x["confidence_mean"] is not None else 0.0), reverse=True)
fn_records.sort(key=lambda x: (x["nearest_prediction_same_label"] is not None, x["label"]), reverse=True)

# Token confusion summaries.
label_confusion = []
for (gold_id, pred_id), count in token_confusion.items():
    label_confusion.append({
        "gold_id": gold_id, "gold_tag": EXPECTED_ID2TAG.get(gold_id, f"ID{gold_id}"),
        "pred_id": pred_id, "pred_tag": model_id2tag.get(pred_id, f"ID{pred_id}"),
        "count": int(count),
    })
label_confusion.sort(key=lambda r: r["count"], reverse=True)

# Actual token-level predictions on supervised positions, normalized as shares.
token_distribution = []
all_supervised = sum(raw_gold_token_counts.values())
for class_id in range(NUM_CLASSES):
    token_distribution.append({
        "id": class_id,
        "tag": model_id2tag.get(class_id, f"ID{class_id}"),
        "gold_supervised_tokens": int(raw_gold_token_counts[class_id]),
        "predicted_on_supervised_tokens": int(raw_pred_token_counts[class_id]),
        "gold_share": raw_gold_token_counts[class_id] / all_supervised if all_supervised else 0.0,
        "predicted_share": raw_pred_token_counts[class_id] / all_supervised if all_supervised else 0.0,
    })

integrity_checks = {
    "checkpoint_bio_map_matches_labels_txt": bool(mapping_matches_labels),
    "manifest_bio_order_matches_labels_txt": bool(manifest_mapping_matches),
    "window_document_ids_match_manifest": bool(window_partition_ok),
    "all_validation_tokens_have_exactly_one_prediction": bool(coverage_ok),
    "checkpoint_f1_reproduces_with_diagnostic_evaluator": bool(f1_reproduces),
    "checkpoint_model_and_head_loaded_strictly": True,
}
structural_issue_keys = [
    "checkpoint_bio_map_matches_labels_txt",
    "manifest_bio_order_matches_labels_txt",
    "window_document_ids_match_manifest",
    "all_validation_tokens_have_exactly_one_prediction",
    "checkpoint_f1_reproduces_with_diagnostic_evaluator",
]
structural_ok = all(integrity_checks[k] for k in structural_issue_keys)
if not structural_ok:
    decision = "FIX MAPPING / PREPROCESSING / EVALUATOR FIRST; do not retrain until these checks pass."
else:
    # The diagnostic is meant to distinguish implementation defects from weak model quality.
    if recomputed_f1 < 0.15 and metrics["predicted_entities"] > 1.25 * max(1, metrics["gold_entities"]):
        decision = "MAPPING/DECODER CHECKS PASS; RETRAIN OR CHANGE THE LEARNING METHOD. Current model is severely over-predicting spans."
    elif recomputed_f1 < 0.15:
        decision = "MAPPING/DECODER CHECKS PASS; RETRAIN OR CHANGE THE LEARNING METHOD. Validation F1 remains very low."
    else:
        decision = "INTEGRITY CHECKS PASS; do not assume mapping bug. Review per-label errors before choosing a targeted retrain."

summary = {
    "checkpoint": str(checkpoint_path),
    "epoch": ckpt.get("epoch"),
    "stored_metrics": stored_metrics,
    "recomputed_metrics": metrics,
    "integrity_checks": integrity_checks,
    "decision": decision,
    "validation_docs": len(val_docs),
    "validation_windows": len(val_data["metadata"]),
    "validation_gold_entities_boundary_aligned_to_tokens": exact_boundary_count,
    "validation_gold_entities_not_exactly_token_boundary_aligned": nonexact_boundary_count,
    "validation_gold_entities_with_empty_surface": invalid_gold_surface_count,
    "invalid_I_transitions": invalid_i_transitions,
    "total_I_tags": all_i_transitions,
    "invalid_I_transition_rate_among_I_tags": invalid_i_transitions / all_i_transitions if all_i_transitions else 0.0,
    "labels_txt_order": ENTITY_LABELS,
    "expected_tag2id": EXPECTED_TAG2ID,
    "checkpoint_tag2id": saved_tag2id,
    "per_document_invalid_I_transitions": invalid_i_by_doc,
    "per_label": metrics["per_label"],
    "top_token_confusions": label_confusion[:100],
    "token_distribution": token_distribution,
    "fp_examples_count": len(fp_records),
    "fn_examples_count": len(fn_records),
}
with (OUT_DIR / "diagnostic_summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
with (OUT_DIR / "fp_examples.jsonl").open("w", encoding="utf-8") as f:
    for row in fp_records:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
with (OUT_DIR / "fn_examples.jsonl").open("w", encoding="utf-8") as f:
    for row in fn_records:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
with (OUT_DIR / "top_token_confusions.json").open("w", encoding="utf-8") as f:
    json.dump(label_confusion[:200], f, ensure_ascii=False, indent=2)

# ------------------------------ Readable report ----------------------------
print("\n" + "=" * 88)
print("VALIDATION EXACT-CHARACTER-SPAN RESULTS")
print("=" * 88)
print(f"Precision: {metrics['strict_macro_precision']:.6f}")
print(f"Recall:    {metrics['strict_macro_recall']:.6f}")
print(f"Macro-F1:  {metrics['strict_macro_f1']:.6f}")
print(f"Gold entities: {metrics['gold_entities']:,} | predicted: {metrics['predicted_entities']:,} | exact: {metrics['exact_matches']:,}")
if stored_f1 is not None:
    print(f"Stored F1: {float(stored_f1):.6f} | difference: {abs(recomputed_f1 - float(stored_f1)):.8f}")
print(f"Gold spans exactly token-boundary aligned: {exact_boundary_count:,} | non-exact: {nonexact_boundary_count:,}")
print(f"Invalid I transitions: {invalid_i_transitions:,}/{all_i_transitions:,} ({100*summary['invalid_I_transition_rate_among_I_tags']:.2f}% of I tags)")
print("\nPer-label metrics (sorted by F1 ascending):")
print(f"{'LABEL':12s} {'GOLD':>6s} {'PRED':>6s} {'TP':>5s} {'FP':>5s} {'FN':>5s} {'PREC':>8s} {'REC':>8s} {'F1':>8s}")
for label, row in sorted(metrics["per_label"].items(), key=lambda kv: kv[1]["f1"]):
    print(f"{label:12s} {row['gold']:6d} {row['predicted']:6d} {row['tp']:5d} {row['fp']:5d} {row['fn']:5d} {row['precision']:8.3f} {row['recall']:8.3f} {row['f1']:8.3f}")
print("\nTop gold->predicted token confusions:")
for row in label_confusion[:20]:
    print(f"gold={row['gold_tag']:<14s} pred={row['pred_tag']:<14s} n={row['count']}")
print("\nRepresentative false positives (max 12):")
for row in fp_records[:12]:
    near = row.get("nearest_gold_same_label")
    print(f"FP {row['label']:<10s} conf={row.get('confidence_mean')} span=({row['start']},{row['end']}) text={row['text'][:110]!r}")
    if near:
        print(f"   nearest same-label gold=({near['start']},{near['end']}) text={near['text'][:100]!r}")
print("\nRepresentative false negatives (max 12):")
for row in fn_records[:12]:
    near = row.get("nearest_prediction_same_label")
    print(f"FN {row['label']:<10s} span=({row['start']},{row['end']}) text={row['text'][:110]!r}")
    if near:
        print(f"   nearest same-label pred=({near['start']},{near['end']}) text={near['text'][:100]!r}")
print("\n" + "=" * 88)
print("DIAGNOSTIC DECISION")
print(decision)
print("=" * 88)
print("Saved outputs:")
print(" -", OUT_DIR / "diagnostic_summary.json")
print(" -", OUT_DIR / "fp_examples.jsonl")
print(" -", OUT_DIR / "fn_examples.jsonl")
print(" -", OUT_DIR / "top_token_confusions.json")
print("Official Fold 3 test data: NOT USED")


ARCH1 + LoRA NER — DIAGNOSTIC (validation split only)
Device: cuda
GPU: Tesla T4
NER checkpoint: /kaggle/working/phase2_arch1_lora/best_arch1_lora_strict_macro_f1.pt
Data directory: /kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data
NER source: /kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER
Tokenizer: /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
Train windows: 502 | shape=(502, 1024) | supervised=377,187
Validation windows: 105 | shape=(105, 1024) | supervised=80,163
Window document split matches manifest: PASS
Labels.txt order: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Checkpoint BIO map matches labels.txt: PASS
Manifest BIO order matches labels.txt: PASS
Strict model + BIO-head restore: PASS | LoRA modules: 48
Checkpoint epoch: 5
Stored validation F1: 0.10702425831617758
All validation tokens receive exactly one prediction: PASS

VALI

In [7]:
#!/usr/bin/env python3
"""Validation-only diagnostic for the Phase-2 Arch1 + LoRA NER checkpoint.

Run from a Kaggle notebook with the Phase-2 dataset mounted. This script:
  1) restores the saved Arch1+LoRA model and BIO head strictly;
  2) audits checkpoint BIO mapping, manifest/window labels, token coverage,
     token-level confusion, BIO transitions, and character offsets;
  3) re-evaluates document-level exact character spans on validation only;
  4) saves per-label metrics and representative FP/FN examples;
  5) recommends mapping/decoder repair versus a model retrain based on checks.

It does not train anything and never reads the official Fold 3 test text.
"""
from __future__ import annotations

import json
import math
import os
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any

import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------ Paths ---------------------------------------
CHECKPOINT_CANDIDATES = [
    Path("/kaggle/working/phase2_arch1_lora/best_arch1_lora_strict_macro_f1.pt"),
    Path("/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch1_lora/best_arch1_lora_strict_macro_f1.pt"),
]
DATA_DIR_CANDIDATES = [
    Path("/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data"),
    Path("/kaggle/working/phase2_arch4_data"),
]
NER_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER"),
    Path("/kaggle/working/ner_data/Phase_2_LNER"),
    Path("/kaggle/input/ner_data/Phase_2_LNER"),
]
TOKENIZER_CANDIDATES = [
    Path("/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"),
    Path("/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/tokenizer.json"),
    Path("/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokens/tokenizer.json"),
]
OUT_DIR = Path("/kaggle/working/phase2_arch1_lora_decoder_sweep")
MAX_LEN = 1024
VOCAB_SIZE = 40000
NUM_CLASSES = 25
IGNORE_INDEX = -100

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"


def first_existing(paths: list[Path], description: str) -> Path:
    for path in paths:
        exists = path.is_file() if path.suffix else path.is_dir()
        if exists:
            return path
    raise FileNotFoundError(
        f"Could not find {description}. Checked:\n" + "\n".join(map(str, paths))
    )


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def normalize_text_map(raw: dict, name: str) -> dict[str, str]:
    out = {}
    for key, value in raw.items():
        if isinstance(value, dict):
            value = value.get("text")
        if not isinstance(value, str):
            raise TypeError(f"{name}[{key!r}] is not text")
        out[str(key)] = value
    return out


def normalize_gold_map(raw: dict, name: str) -> dict[str, list]:
    out = {str(key): value for key, value in raw.items()}
    for key, entities in out.items():
        if not isinstance(entities, list):
            raise TypeError(f"{name}[{key!r}] must be a list")
    return out


def read_entity(entity: Any) -> tuple[int, int, str]:
    """Support the dataset's [start, end, label] form and common dict variants."""
    if isinstance(entity, dict):
        start = entity.get("start", entity.get("start_char", entity.get("begin")))
        end = entity.get("end", entity.get("end_char", entity.get("stop")))
        label = entity.get("label", entity.get("type", entity.get("entity_type")))
        if start is None or end is None or label is None:
            raise ValueError(f"Unrecognized entity dictionary: {entity}")
        return int(start), int(end), str(label)
    if not isinstance(entity, (list, tuple)) or len(entity) < 3:
        raise ValueError(f"Unrecognized entity item: {entity!r}")
    return int(entity[0]), int(entity[1]), str(entity[2])


def correct_offsets(text: str, raw_offsets: list[tuple[int, int]]) -> list[tuple[int, int]]:
    """Exact same offset correction used during Arch1+LoRA training/evaluation."""
    corrected = []
    for start, end in raw_offsets:
        start, end = int(start), int(end)
        if not (0 <= start <= end <= len(text)):
            raise ValueError(f"Invalid offset {(start, end)} for text length {len(text)}")
        while start < end and text[start].isspace():
            start += 1
        corrected.append((start, end))
    return corrected


# --------------------------- Arch1 + LoRA model -----------------------------
class Arch1CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, block_size: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("mask", mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq_len, channels = x.shape
        q = self.q_proj(x).view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.mask[:, :, :seq_len, :seq_len] == 0, float("-inf"))
        attn = F.softmax(scores, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(batch, seq_len, channels)
        return self.out_proj(out)


class Arch1FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(F.gelu(self.fc1(x)))


class Arch1TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, block_size: int):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = Arch1CausalSelfAttention(d_model, n_heads, block_size)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = Arch1FeedForward(d_model, d_ff)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class Arch1LegalDecoderLM(nn.Module):
    def __init__(self, vocab_size=40000, block_size=1024, d_model=768,
                 n_layers=8, n_heads=8, d_ff=3072):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([
            Arch1TransformerBlock(d_model, n_heads, d_ff, block_size)
            for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

    def forward_hidden(self, input_ids: torch.Tensor) -> torch.Tensor:
        batch, seq_len = input_ids.shape
        if seq_len > self.block_size:
            raise ValueError(f"Sequence length {seq_len} exceeds {self.block_size}")
        positions = torch.arange(seq_len, device=input_ids.device)
        x = self.token_emb(input_ids) + self.pos_emb(positions)
        for block in self.blocks:
            x = block(x)
        return self.final_norm(x)


class LoRALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, rank=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base_layer
        for p in self.base.parameters():
            p.requires_grad = False
        self.rank = int(rank)
        self.alpha = float(alpha)
        self.scaling = self.alpha / self.rank
        self.dropout = nn.Dropout(dropout)
        self.lora_A = nn.Linear(base_layer.in_features, self.rank, bias=False)
        self.lora_B = nn.Linear(self.rank, base_layer.out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.base(x) + self.scaling * self.lora_B(self.lora_A(self.dropout(x)))


def inject_lora(model: Arch1LegalDecoderLM) -> int:
    targets = [
        ("attn", "q_proj"), ("attn", "k_proj"),
        ("attn", "v_proj"), ("attn", "out_proj"),
        ("mlp", "fc1"), ("mlp", "fc2"),
    ]
    count = 0
    for block in model.blocks:
        for parent_name, attr in targets:
            parent = getattr(block, parent_name)
            old = getattr(parent, attr)
            setattr(parent, attr, LoRALinear(old, rank=8, alpha=16, dropout=0.05))
            count += 1
    return count


def decode_bio_spans(tag_ids: list[int], offsets: list[tuple[int, int]], id2tag: dict[int, str]):
    """Same decoder behavior as training: a standalone I-X starts a new X span."""
    if len(tag_ids) != len(offsets):
        raise ValueError(f"tag/offset length mismatch: {len(tag_ids)} vs {len(offsets)}")
    spans = []
    readable = ["O" if s >= e else id2tag[int(t)] for t, (s, e) in zip(tag_ids, offsets)]
    i = 0
    while i < len(readable):
        tag = readable[i]
        if tag == "O":
            i += 1
            continue
        if not (tag.startswith("B-") or tag.startswith("I-")):
            i += 1
            continue
        label = tag[2:]
        start_char = offsets[i][0]
        j = i + 1
        while j < len(readable) and readable[j] == f"I-{label}":
            j += 1
        end_char = offsets[j - 1][1]
        if start_char < end_char:
            spans.append((start_char, end_char, label, i, j))
        i = j
    return spans, readable


def macro_metrics(gold: Counter, pred: Counter, labels: list[str]) -> dict:
    matched = gold & pred
    tp_by_label = Counter()
    for span, count in matched.items():
        tp_by_label[span[3]] += count
    per_label = {}
    for label in labels:
        gold_n = sum(n for sp, n in gold.items() if sp[3] == label)
        pred_n = sum(n for sp, n in pred.items() if sp[3] == label)
        tp = tp_by_label[label]
        fp, fn = pred_n - tp, gold_n - tp
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        per_label[label] = {
            "gold": int(gold_n), "predicted": int(pred_n), "tp": int(tp),
            "fp": int(fp), "fn": int(fn), "precision": p, "recall": r, "f1": f,
        }
    return {
        "strict_macro_precision": sum(v["precision"] for v in per_label.values()) / len(labels),
        "strict_macro_recall": sum(v["recall"] for v in per_label.values()) / len(labels),
        "strict_macro_f1": sum(v["f1"] for v in per_label.values()) / len(labels),
        "gold_entities": int(sum(gold.values())),
        "predicted_entities": int(sum(pred.values())),
        "exact_matches": int(sum(matched.values())),
        "per_label": per_label,
    }


def safe_surface(text: str, start: int, end: int, radius: int = 80) -> str:
    left, right = max(0, start - radius), min(len(text), end + radius)
    return text[left:right].replace("\n", " ").replace("\r", " ")


def constrained_bio_viterbi(logit_matrix: torch.Tensor, offsets: list[tuple[int, int]],
                            id2tag: dict[int, str], entity_margin: float = 0.0) -> list[int]:
    """Decode legal IOB2 paths and optionally subtract a margin from all entity logits.

    An I-X tag may only follow B-X or I-X. `entity_margin > 0` makes O relatively
    more competitive and is useful to test the current model's over-prediction.
    This is a validation-only decoder sweep; it does not update model weights.
    """
    import numpy as np

    emissions = logit_matrix.detach().cpu().float().numpy().copy()
    n_tokens, n_tags = emissions.shape
    if len(offsets) != n_tokens or n_tags != NUM_CLASSES:
        raise ValueError(f"Bad Viterbi shapes: logits={emissions.shape}, offsets={len(offsets)}")
    if n_tokens == 0:
        return []

    # Reduce all non-O scores equally; the sweep tests whether entity over-prediction
    # is primarily a decision-margin problem without changing class identities.
    emissions[:, 1:] -= float(entity_margin)

    tag_strings = [id2tag[i] for i in range(n_tags)]
    parsed = []
    for tag in tag_strings:
        if tag == "O":
            parsed.append(("O", ""))
        elif tag.startswith("B-") or tag.startswith("I-"):
            parsed.append((tag[:1], tag[2:]))
        else:
            raise ValueError(f"Unknown BIO tag: {tag}")

    neg = -1e9
    trans = np.zeros((n_tags, n_tags), dtype=np.float32)  # prev tag -> current tag
    for cur_id, (prefix, label) in enumerate(parsed):
        if prefix == "I":
            for prev_id, (prev_prefix, prev_label) in enumerate(parsed):
                if not (prev_label == label and prev_prefix in ("B", "I")):
                    trans[prev_id, cur_id] = neg

    # Empty-offset special/padding tokens are forced to O.
    for t, (start, end) in enumerate(offsets):
        if start >= end:
            emissions[t, :] = neg
            emissions[t, 0] = 0.0

    dp = emissions[0].copy()
    for tag_id, (prefix, _) in enumerate(parsed):
        if prefix == "I":
            dp[tag_id] = neg  # A sequence cannot begin with I-X under IOB2.
    backpointers = np.zeros((n_tokens, n_tags), dtype=np.int16)
    for t in range(1, n_tokens):
        scores = dp[:, None] + trans
        best_prev = scores.argmax(axis=0)
        dp = emissions[t] + scores[best_prev, np.arange(n_tags)]
        backpointers[t] = best_prev.astype(np.int16)
    best = int(dp.argmax())
    path = [0] * n_tokens
    path[-1] = best
    for t in range(n_tokens - 1, 0, -1):
        path[t - 1] = int(backpointers[t, path[t]])
    return path


def evaluate_decoded_tags(tag_paths: dict[str, list[int]], offsets_by_doc: dict[str, list[tuple[int, int]],],
                          val_by_id: dict[str, dict], gold_spans: Counter,
                          id2tag: dict[int, str], labels: list[str]) -> tuple[dict, Counter]:
    pred = Counter()
    for doc_id, tag_ids in tag_paths.items():
        spans, _ = decode_bio_spans(tag_ids, offsets_by_doc[doc_id], id2tag)
        for start, end, label, _, _ in spans:
            pred[(doc_id, start, end, label)] += 1
    return macro_metrics(gold_spans, pred, labels), pred


# ------------------------------ Load data -----------------------------------
checkpoint_path = first_existing(CHECKPOINT_CANDIDATES, "Arch1+LoRA NER checkpoint")
data_dir = first_existing(DATA_DIR_CANDIDATES, "Phase-2 cached-window directory")
ner_root = first_existing(NER_ROOT_CANDIDATES, "Phase-2 NER source directory")
tokenizer_path = first_existing(TOKENIZER_CANDIDATES, "Phase-1 tokenizer.json")

train_windows_path = data_dir / "train_windows.pt"
val_windows_path = data_dir / "validation_windows.pt"
manifest_path = data_dir / "split_manifest.json"
for p in [train_windows_path, val_windows_path, manifest_path]:
    if not p.is_file():
        raise FileNotFoundError(f"Missing required file: {p}")

print("=" * 88)
print("ARCH1 + LoRA NER — DIAGNOSTIC (validation split only)")
print("=" * 88)
print("Device:", DEVICE)
if USE_AMP:
    print("GPU:", torch.cuda.get_device_name(0))
print("NER checkpoint:", checkpoint_path)
print("Data directory:", data_dir)
print("NER source:", ner_root)
print("Tokenizer:", tokenizer_path)

from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(tokenizer_path))
with (ner_root / "Train/labels.txt").open("r", encoding="utf-8") as f:
    ENTITY_LABELS = [line.strip() for line in f if line.strip()]
if len(ENTITY_LABELS) != 12:
    raise ValueError(f"Expected 12 entity labels, found {len(ENTITY_LABELS)}: {ENTITY_LABELS}")
EXPECTED_BIO_TAGS = ["O"] + [f"B-{x}" for x in ENTITY_LABELS] + [f"I-{x}" for x in ENTITY_LABELS]
EXPECTED_TAG2ID = {tag: i for i, tag in enumerate(EXPECTED_BIO_TAGS)}
EXPECTED_ID2TAG = {i: tag for tag, i in EXPECTED_TAG2ID.items()}

f1_text = normalize_text_map(load_json(ner_root / "Train/fold1-text.json"), "fold1_text")
f1_gold = normalize_gold_map(load_json(ner_root / "Train/fold1-gold.json"), "fold1_gold")
f2_text = normalize_text_map(load_json(ner_root / "Train/fold2-text.json"), "fold2_text")
f2_gold = normalize_gold_map(load_json(ner_root / "Train/fold2-gold.json"), "fold2_gold")
if set(f1_text) != set(f1_gold) or set(f2_text) != set(f2_gold):
    raise ValueError("Source text IDs and gold IDs do not align within a fold")
if set(f1_text) & set(f2_text):
    raise ValueError("Duplicate document IDs appear across folds")
all_docs: dict[str, dict] = {}
for texts, golds in [(f1_text, f1_gold), (f2_text, f2_gold)]:
    for doc_id, text in texts.items():
        all_docs[doc_id] = {"text": text, "entities": golds[doc_id]}
if len(all_docs) != 70:
    raise ValueError(f"Expected exactly 70 labelled docs, found {len(all_docs)}")

manifest = load_json(manifest_path)
train_ids = [str(x) for x in manifest["train_doc_ids"]]
val_ids = [str(x) for x in manifest["validation_doc_ids"]]
if len(train_ids) != 56 or len(val_ids) != 14:
    raise ValueError(f"Expected 56/14 train/val docs, found {len(train_ids)}/{len(val_ids)}")
if set(train_ids) & set(val_ids) or (set(train_ids) | set(val_ids)) != set(all_docs):
    raise ValueError("Split manifest does not partition the 70 labelled documents exactly")
val_docs = {doc_id: all_docs[doc_id] for doc_id in val_ids}

train_data = torch.load(train_windows_path, map_location="cpu", weights_only=False)
val_data = torch.load(val_windows_path, map_location="cpu", weights_only=False)
for name, data in [("train", train_data), ("validation", val_data)]:
    x, y, metadata = data["input_ids"], data["labels"], data["metadata"]
    assert x.ndim == y.ndim == 2 and x.shape == y.shape
    assert x.shape[1] == MAX_LEN and x.dtype == torch.long and y.dtype == torch.long
    assert len(metadata) == x.shape[0]
    active = y[y != IGNORE_INDEX]
    assert active.numel() > 0 and int(active.min()) >= 0 and int(active.max()) < NUM_CLASSES
    print(f"{name.title()} windows: {len(metadata)} | shape={tuple(x.shape)} | supervised={int(active.numel()):,}")

train_window_ids = {str(m["doc_id"]) for m in train_data["metadata"]}
val_window_ids = {str(m["doc_id"]) for m in val_data["metadata"]}
window_partition_ok = (train_window_ids == set(train_ids) and val_window_ids == set(val_ids)
                       and not (train_window_ids & val_window_ids))
print("Window document split matches manifest:", "PASS" if window_partition_ok else "FAIL")

# --------------------------- Check checkpoint mapping ----------------------
ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
if not isinstance(ckpt, dict) or "model" not in ckpt or "bio_head" not in ckpt:
    raise ValueError("This does not look like the saved Arch1+LoRA NER checkpoint (missing model/bio_head)")
saved_tag2id_raw = ckpt.get("tag2id")
saved_tag2id = None
if isinstance(saved_tag2id_raw, dict):
    saved_tag2id = {str(k): int(v) for k, v in saved_tag2id_raw.items()}
else:
    print("WARNING: checkpoint does not store tag2id; using labels.txt ordering for diagnostic decode.")

mapping_matches_labels = saved_tag2id is not None and saved_tag2id == EXPECTED_TAG2ID
manifest_tags = manifest.get("bio_tags")
manifest_mapping_matches = manifest_tags is None or list(manifest_tags) == EXPECTED_BIO_TAGS
# Decode using the mapping that the model was actually trained with if available.
if saved_tag2id is not None:
    model_id2tag = {idx: tag for tag, idx in saved_tag2id.items()}
    if len(model_id2tag) != NUM_CLASSES or set(model_id2tag) != set(range(NUM_CLASSES)):
        raise ValueError(f"Checkpoint tag2id is not a complete 25-class mapping: {saved_tag2id}")
else:
    model_id2tag = EXPECTED_ID2TAG

print("Labels.txt order:", ENTITY_LABELS)
print("Checkpoint BIO map matches labels.txt:", "PASS" if mapping_matches_labels else "FAIL")
print("Manifest BIO order matches labels.txt:", "PASS" if manifest_mapping_matches else "FAIL")
if saved_tag2id is not None and not mapping_matches_labels:
    print("Expected mapping:", EXPECTED_TAG2ID)
    print("Checkpoint mapping:", saved_tag2id)
if not manifest_mapping_matches:
    print("Manifest BIO tags:", manifest_tags)

# -------------------------- Restore model strictly -------------------------
model = Arch1LegalDecoderLM(vocab_size=VOCAB_SIZE, block_size=MAX_LEN)
inserted = inject_lora(model)
if inserted != 48:
    raise AssertionError(f"Expected 48 LoRA modules, got {inserted}")
missing, unexpected = model.load_state_dict(ckpt["model"], strict=True)
# strict=True raises if mismatched; retain this assertion for readability.
if missing or unexpected:
    raise RuntimeError(f"Checkpoint state mismatch: missing={missing}, unexpected={unexpected}")
head = nn.Linear(768, NUM_CLASSES)
head.load_state_dict(ckpt["bio_head"], strict=True)
model = model.to(DEVICE).eval()
head = head.to(DEVICE).eval()
print("Strict model + BIO-head restore: PASS | LoRA modules:", inserted)
print("Checkpoint epoch:", ckpt.get("epoch"))
print("Stored validation F1:", (ckpt.get("metrics") or {}).get("strict_macro_f1"))

# -------------------------- Run validation inference ------------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)
val_by_id = {str(k): v for k, v in val_docs.items()}
offsets_by_doc: dict[str, list[tuple[int, int]]] = {}
pred_tags_by_doc: dict[str, list[int]] = {}
pred_conf_by_doc: dict[str, list[float]] = {}
logits_by_doc: dict[str, torch.Tensor] = {}
ownership_by_doc: dict[str, list[bool]] = {}

for doc_id, item in val_by_id.items():
    enc = tokenizer.encode(item["text"])
    offsets = correct_offsets(item["text"], enc.offsets)
    offsets_by_doc[doc_id] = offsets
    pred_tags_by_doc[doc_id] = [0] * len(offsets)
    pred_conf_by_doc[doc_id] = [0.0] * len(offsets)
    # Store full 25-way logits, allowing decoder-only alternatives without retraining.
    logits_by_doc[doc_id] = torch.zeros((len(offsets), NUM_CLASSES), dtype=torch.float32)
    ownership_by_doc[doc_id] = [False] * len(offsets)

# Token-label confusion is calculated from supervised validation positions in cached windows.
token_confusion = Counter()
raw_pred_token_counts = Counter()
raw_gold_token_counts = Counter()
invalid_i_transitions = 0
all_i_transitions = 0
max_conf_by_doc = pred_conf_by_doc

with torch.inference_mode():
    for wi, meta in enumerate(val_data["metadata"]):
        doc_id = str(meta["doc_id"])
        if doc_id not in offsets_by_doc:
            raise KeyError(f"Window references unknown validation doc: {doc_id}")
        x = val_data["input_ids"][wi:wi+1].to(DEVICE)
        y = val_data["labels"][wi].tolist()
        if USE_AMP:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = head(model.forward_hidden(x))
        else:
            logits = head(model.forward_hidden(x))
        logits = logits.float()[0]
        probs, pred_ids = torch.softmax(logits, dim=-1).max(dim=-1)
        pred_ids = pred_ids.cpu().tolist()
        probs = probs.cpu().tolist()

        for pos, gold_id in enumerate(y):
            if gold_id == IGNORE_INDEX:
                continue
            pred_id = int(pred_ids[pos])
            token_confusion[(int(gold_id), pred_id)] += 1
            raw_pred_token_counts[pred_id] += 1
            raw_gold_token_counts[int(gold_id)] += 1

        window_start = int(meta["window_start"])
        valid_length = int(meta["valid_length"])
        owner_local_start = int(meta["owner_local_start"])
        if not (0 <= owner_local_start <= valid_length <= MAX_LEN):
            raise ValueError(f"Bad window bounds in metadata row {wi}: {meta}")
        for local_idx in range(owner_local_start, valid_length):
            global_idx = window_start + local_idx
            if global_idx >= len(offsets_by_doc[doc_id]):
                raise IndexError(f"Token outside document {doc_id}: {global_idx}")
            if ownership_by_doc[doc_id][global_idx]:
                raise AssertionError(f"Duplicate token owner: {doc_id}/{global_idx}")
            pred_tags_by_doc[doc_id][global_idx] = int(pred_ids[local_idx])
            pred_conf_by_doc[doc_id][global_idx] = float(probs[local_idx])
            logits_by_doc[doc_id][global_idx] = logits[local_idx].detach().cpu()
            ownership_by_doc[doc_id][global_idx] = True

coverage_ok = True
coverage_missing = {}
for doc_id, ownership in ownership_by_doc.items():
    missing_n = ownership.count(False)
    if missing_n:
        coverage_ok = False
        coverage_missing[doc_id] = missing_n
print("All validation tokens receive exactly one prediction:", "PASS" if coverage_ok else "FAIL")
if coverage_missing:
    print("Missing ownership counts:", coverage_missing)

# Validate that current model's token IDs can be interpreted as BIO tags.
for doc_id, pred_ids in pred_tags_by_doc.items():
    for tag_id in pred_ids:
        if tag_id not in model_id2tag:
            raise ValueError(f"Predicted class ID {tag_id} not in checkpoint BIO mapping")

# Gold and predicted exact character spans, plus confidence for debugging.
gold_spans = Counter()
pred_spans = Counter()
pred_span_details = []
invalid_i_by_doc = {}
for doc_id, item in val_by_id.items():
    text = item["text"]
    offsets = offsets_by_doc[doc_id]
    tags = pred_tags_by_doc[doc_id]
    if len(offsets) != len(tags):
        raise AssertionError(f"Tokenizer/prediction length mismatch: {doc_id}: {len(offsets)} != {len(tags)}")
    for ent in item["entities"]:
        start, end, label = read_entity(ent)
        if not (0 <= start <= end <= len(text)):
            raise ValueError(f"Gold span out of bounds in {doc_id}: {(start, end, label)}")
        gold_spans[(doc_id, start, end, label)] += 1

    readable_tags = []
    for tag_id, (start, end) in zip(tags, offsets):
        readable_tags.append("O" if start >= end else model_id2tag[int(tag_id)])
    bad_i = 0
    i_tags = 0
    for i, tag in enumerate(readable_tags):
        if tag.startswith("I-"):
            i_tags += 1
            label = tag[2:]
            if i == 0 or readable_tags[i-1] not in (f"B-{label}", f"I-{label}"):
                bad_i += 1
    invalid_i_transitions += bad_i
    all_i_transitions += i_tags
    invalid_i_by_doc[doc_id] = {"invalid_I_transitions": bad_i, "I_tags": i_tags}

    spans, _ = decode_bio_spans(tags, offsets, model_id2tag)
    for start, end, label, token_start, token_end in spans:
        key = (doc_id, start, end, label)
        pred_spans[key] += 1
        confs = pred_conf_by_doc[doc_id][token_start:token_end]
        pred_span_details.append({
            "doc_id": doc_id, "start": start, "end": end, "label": label,
            "confidence_mean": (sum(confs) / len(confs)) if confs else 0.0,
            "confidence_min": min(confs) if confs else 0.0,
            "text": text[start:end],
        })

metrics = macro_metrics(gold_spans, pred_spans, ENTITY_LABELS)

# ----------------------- Decoder-only validation sweep ----------------------
# Compare the original checkpoint decoder with legal IOB2 Viterbi decoding.
# Positive margins reduce all entity logits relative to O to test whether the
# observed false-positive excess can be mitigated without retraining.
decoder_sweep = [{
    "decoder": "current_argmax_permissive_bio",
    "entity_margin": None,
    "strict_macro_precision": metrics["strict_macro_precision"],
    "strict_macro_recall": metrics["strict_macro_recall"],
    "strict_macro_f1": metrics["strict_macro_f1"],
    "gold_entities": metrics["gold_entities"],
    "predicted_entities": metrics["predicted_entities"],
    "exact_matches": metrics["exact_matches"],
    "per_label": metrics["per_label"],
}]
for margin in [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]:
    viterbi_paths = {}
    for doc_id, logits_doc in logits_by_doc.items():
        viterbi_paths[doc_id] = constrained_bio_viterbi(
            logits_doc, offsets_by_doc[doc_id], model_id2tag, entity_margin=margin
        )
    v_metrics, _ = evaluate_decoded_tags(
        viterbi_paths, offsets_by_doc, val_by_id, gold_spans, model_id2tag, ENTITY_LABELS
    )
    decoder_sweep.append({
        "decoder": "constrained_iob2_viterbi",
        "entity_margin": margin,
        "strict_macro_precision": v_metrics["strict_macro_precision"],
        "strict_macro_recall": v_metrics["strict_macro_recall"],
        "strict_macro_f1": v_metrics["strict_macro_f1"],
        "gold_entities": v_metrics["gold_entities"],
        "predicted_entities": v_metrics["predicted_entities"],
        "exact_matches": v_metrics["exact_matches"],
        "per_label": v_metrics["per_label"],
    })

best_decoder = max(decoder_sweep, key=lambda row: row["strict_macro_f1"])
with (OUT_DIR / "decoder_sweep.json").open("w", encoding="utf-8") as f:
    json.dump({
        "validation_only": True,
        "official_fold3_used": False,
        "note": "Decoder/margin selection on this validation split may overfit; do not report as official test performance.",
        "results_sorted_by_f1": sorted(decoder_sweep, key=lambda row: row["strict_macro_f1"], reverse=True),
        "best_decoder": best_decoder,
    }, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 88)
print("DECODER-ONLY SWEEP (VALIDATION ONLY; MODEL WEIGHTS UNCHANGED)")
print("=" * 88)
for row in sorted(decoder_sweep, key=lambda r: r["strict_macro_f1"], reverse=True):
    margin = row["entity_margin"]
    margin_text = "n/a" if margin is None else f"{margin:.2f}"
    print(f"{row['decoder']:<32s} margin={margin_text:>4s} "
          f"P={row['strict_macro_precision']:.6f} "
          f"R={row['strict_macro_recall']:.6f} "
          f"F1={row['strict_macro_f1']:.6f} "
          f"pred={row['predicted_entities']} exact={row['exact_matches']}")
print("Best validation-only decoding candidate:", best_decoder["decoder"],
      "margin=", best_decoder["entity_margin"],
      "F1=", f"{best_decoder['strict_macro_f1']:.6f}")
print("Decoder sweep JSON:", OUT_DIR / "decoder_sweep.json")

stored_metrics = ckpt.get("metrics") or {}
recomputed_f1 = float(metrics["strict_macro_f1"])
stored_f1 = stored_metrics.get("strict_macro_f1")
f1_reproduces = stored_f1 is not None and abs(recomputed_f1 - float(stored_f1)) <= 1e-4

# Offset/alignment diagnostics on validation gold entities.
exact_boundary_count = 0
nonexact_boundary_count = 0
invalid_gold_surface_count = 0
for doc_id, item in val_by_id.items():
    text = item["text"]
    offsets = offsets_by_doc[doc_id]
    token_starts = {s for s, e in offsets if s < e}
    token_ends = {e for s, e in offsets if s < e}
    for ent in item["entities"]:
        start, end, label = read_entity(ent)
        if not text[start:end]:
            invalid_gold_surface_count += 1
        if start in token_starts and end in token_ends:
            exact_boundary_count += 1
        else:
            nonexact_boundary_count += 1

# Save a list of exact FP/FN examples and close same-label overlaps for diagnosis.
matched = gold_spans & pred_spans
fp_keys = list((pred_spans - gold_spans).elements())
fn_keys = list((gold_spans - pred_spans).elements())

# Build per-label output and examples, with overlapping gold candidates to reveal boundary-only misses.
def nearest_gold_same_label(doc_id: str, start: int, end: int, label: str):
    cands = []
    for gdoc, gs, ge, glabel in gold_spans:
        if gdoc != doc_id or glabel != label:
            continue
        overlap = max(0, min(end, ge) - max(start, gs))
        distance = max(0, max(gs - end, start - ge))
        if overlap > 0 or distance <= 20:
            cands.append((overlap, -distance, gs, ge, glabel))
    cands.sort(reverse=True)
    if not cands:
        return None
    _, _, gs, ge, glabel = cands[0]
    return {"start": gs, "end": ge, "label": glabel, "text": val_by_id[doc_id]["text"][gs:ge]}

fp_records = []
for key in fp_keys:
    doc_id, start, end, label = key
    text = val_by_id[doc_id]["text"]
    matching_detail = next((d for d in pred_span_details if (d["doc_id"], d["start"], d["end"], d["label"]) == key), {})
    fp_records.append({
        "kind": "FP", "doc_id": doc_id, "start": start, "end": end, "label": label,
        "text": text[start:end], "context": safe_surface(text, start, end),
        "nearest_gold_same_label": nearest_gold_same_label(doc_id, start, end, label),
        "confidence_mean": matching_detail.get("confidence_mean"),
    })
fn_records = []
for key in fn_keys:
    doc_id, start, end, label = key
    text = val_by_id[doc_id]["text"]
    # Find best overlapping/well-near prediction of same label.
    cands = []
    for pdoc, ps, pe, plabel in pred_spans:
        if pdoc != doc_id or plabel != label:
            continue
        overlap = max(0, min(end, pe) - max(start, ps))
        distance = max(0, max(ps - end, start - pe))
        if overlap > 0 or distance <= 20:
            cands.append((overlap, -distance, ps, pe, plabel))
    cands.sort(reverse=True)
    nearest_pred = None
    if cands:
        _, _, ps, pe, plabel = cands[0]
        nearest_pred = {"start": ps, "end": pe, "label": plabel, "text": text[ps:pe]}
    fn_records.append({
        "kind": "FN", "doc_id": doc_id, "start": start, "end": end, "label": label,
        "text": text[start:end], "context": safe_surface(text, start, end),
        "nearest_prediction_same_label": nearest_pred,
    })

# Stable ordering: examples with nearest boundary overlap first, then label.
fp_records.sort(key=lambda x: (x["nearest_gold_same_label"] is not None,
                               x["confidence_mean"] if x["confidence_mean"] is not None else 0.0), reverse=True)
fn_records.sort(key=lambda x: (x["nearest_prediction_same_label"] is not None, x["label"]), reverse=True)

# Token confusion summaries.
label_confusion = []
for (gold_id, pred_id), count in token_confusion.items():
    label_confusion.append({
        "gold_id": gold_id, "gold_tag": EXPECTED_ID2TAG.get(gold_id, f"ID{gold_id}"),
        "pred_id": pred_id, "pred_tag": model_id2tag.get(pred_id, f"ID{pred_id}"),
        "count": int(count),
    })
label_confusion.sort(key=lambda r: r["count"], reverse=True)

# Actual token-level predictions on supervised positions, normalized as shares.
token_distribution = []
all_supervised = sum(raw_gold_token_counts.values())
for class_id in range(NUM_CLASSES):
    token_distribution.append({
        "id": class_id,
        "tag": model_id2tag.get(class_id, f"ID{class_id}"),
        "gold_supervised_tokens": int(raw_gold_token_counts[class_id]),
        "predicted_on_supervised_tokens": int(raw_pred_token_counts[class_id]),
        "gold_share": raw_gold_token_counts[class_id] / all_supervised if all_supervised else 0.0,
        "predicted_share": raw_pred_token_counts[class_id] / all_supervised if all_supervised else 0.0,
    })

integrity_checks = {
    "checkpoint_bio_map_matches_labels_txt": bool(mapping_matches_labels),
    "manifest_bio_order_matches_labels_txt": bool(manifest_mapping_matches),
    "window_document_ids_match_manifest": bool(window_partition_ok),
    "all_validation_tokens_have_exactly_one_prediction": bool(coverage_ok),
    "checkpoint_f1_reproduces_with_diagnostic_evaluator": bool(f1_reproduces),
    "checkpoint_model_and_head_loaded_strictly": True,
}
structural_issue_keys = [
    "checkpoint_bio_map_matches_labels_txt",
    "manifest_bio_order_matches_labels_txt",
    "window_document_ids_match_manifest",
    "all_validation_tokens_have_exactly_one_prediction",
    "checkpoint_f1_reproduces_with_diagnostic_evaluator",
]
structural_ok = all(integrity_checks[k] for k in structural_issue_keys)
if not structural_ok:
    decision = "FIX MAPPING / PREPROCESSING / EVALUATOR FIRST; do not retrain until these checks pass."
else:
    # The diagnostic is meant to distinguish implementation defects from weak model quality.
    if recomputed_f1 < 0.15 and metrics["predicted_entities"] > 1.25 * max(1, metrics["gold_entities"]):
        decision = "MAPPING/DECODER CHECKS PASS; RETRAIN OR CHANGE THE LEARNING METHOD. Current model is severely over-predicting spans."
    elif recomputed_f1 < 0.15:
        decision = "MAPPING/DECODER CHECKS PASS; RETRAIN OR CHANGE THE LEARNING METHOD. Validation F1 remains very low."
    else:
        decision = "INTEGRITY CHECKS PASS; do not assume mapping bug. Review per-label errors before choosing a targeted retrain."

summary = {
    "checkpoint": str(checkpoint_path),
    "epoch": ckpt.get("epoch"),
    "stored_metrics": stored_metrics,
    "recomputed_metrics": metrics,
    "integrity_checks": integrity_checks,
    "decision": decision,
    "validation_docs": len(val_docs),
    "validation_windows": len(val_data["metadata"]),
    "validation_gold_entities_boundary_aligned_to_tokens": exact_boundary_count,
    "validation_gold_entities_not_exactly_token_boundary_aligned": nonexact_boundary_count,
    "validation_gold_entities_with_empty_surface": invalid_gold_surface_count,
    "invalid_I_transitions": invalid_i_transitions,
    "total_I_tags": all_i_transitions,
    "invalid_I_transition_rate_among_I_tags": invalid_i_transitions / all_i_transitions if all_i_transitions else 0.0,
    "labels_txt_order": ENTITY_LABELS,
    "expected_tag2id": EXPECTED_TAG2ID,
    "checkpoint_tag2id": saved_tag2id,
    "per_document_invalid_I_transitions": invalid_i_by_doc,
    "per_label": metrics["per_label"],
    "top_token_confusions": label_confusion[:100],
    "token_distribution": token_distribution,
    "fp_examples_count": len(fp_records),
    "fn_examples_count": len(fn_records),
}
with (OUT_DIR / "diagnostic_summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
with (OUT_DIR / "fp_examples.jsonl").open("w", encoding="utf-8") as f:
    for row in fp_records:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
with (OUT_DIR / "fn_examples.jsonl").open("w", encoding="utf-8") as f:
    for row in fn_records:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
with (OUT_DIR / "top_token_confusions.json").open("w", encoding="utf-8") as f:
    json.dump(label_confusion[:200], f, ensure_ascii=False, indent=2)

# ------------------------------ Readable report ----------------------------
print("\n" + "=" * 88)
print("VALIDATION EXACT-CHARACTER-SPAN RESULTS")
print("=" * 88)
print(f"Precision: {metrics['strict_macro_precision']:.6f}")
print(f"Recall:    {metrics['strict_macro_recall']:.6f}")
print(f"Macro-F1:  {metrics['strict_macro_f1']:.6f}")
print(f"Gold entities: {metrics['gold_entities']:,} | predicted: {metrics['predicted_entities']:,} | exact: {metrics['exact_matches']:,}")
if stored_f1 is not None:
    print(f"Stored F1: {float(stored_f1):.6f} | difference: {abs(recomputed_f1 - float(stored_f1)):.8f}")
print(f"Gold spans exactly token-boundary aligned: {exact_boundary_count:,} | non-exact: {nonexact_boundary_count:,}")
print(f"Invalid I transitions: {invalid_i_transitions:,}/{all_i_transitions:,} ({100*summary['invalid_I_transition_rate_among_I_tags']:.2f}% of I tags)")
print("\nPer-label metrics (sorted by F1 ascending):")
print(f"{'LABEL':12s} {'GOLD':>6s} {'PRED':>6s} {'TP':>5s} {'FP':>5s} {'FN':>5s} {'PREC':>8s} {'REC':>8s} {'F1':>8s}")
for label, row in sorted(metrics["per_label"].items(), key=lambda kv: kv[1]["f1"]):
    print(f"{label:12s} {row['gold']:6d} {row['predicted']:6d} {row['tp']:5d} {row['fp']:5d} {row['fn']:5d} {row['precision']:8.3f} {row['recall']:8.3f} {row['f1']:8.3f}")
print("\nTop gold->predicted token confusions:")
for row in label_confusion[:20]:
    print(f"gold={row['gold_tag']:<14s} pred={row['pred_tag']:<14s} n={row['count']}")
print("\nRepresentative false positives (max 12):")
for row in fp_records[:12]:
    near = row.get("nearest_gold_same_label")
    print(f"FP {row['label']:<10s} conf={row.get('confidence_mean')} span=({row['start']},{row['end']}) text={row['text'][:110]!r}")
    if near:
        print(f"   nearest same-label gold=({near['start']},{near['end']}) text={near['text'][:100]!r}")
print("\nRepresentative false negatives (max 12):")
for row in fn_records[:12]:
    near = row.get("nearest_prediction_same_label")
    print(f"FN {row['label']:<10s} span=({row['start']},{row['end']}) text={row['text'][:110]!r}")
    if near:
        print(f"   nearest same-label pred=({near['start']},{near['end']}) text={near['text'][:100]!r}")
print("\n" + "=" * 88)
print("DIAGNOSTIC DECISION")
print(decision)
print("=" * 88)
print("Saved outputs:")
print(" -", OUT_DIR / "diagnostic_summary.json")
print(" -", OUT_DIR / "fp_examples.jsonl")
print(" -", OUT_DIR / "fn_examples.jsonl")
print(" -", OUT_DIR / "top_token_confusions.json")
print("Official Fold 3 test data: NOT USED")


ARCH1 + LoRA NER — DIAGNOSTIC (validation split only)
Device: cuda
GPU: Tesla T4
NER checkpoint: /kaggle/working/phase2_arch1_lora/best_arch1_lora_strict_macro_f1.pt
Data directory: /kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data
NER source: /kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER
Tokenizer: /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
Train windows: 502 | shape=(502, 1024) | supervised=377,187
Validation windows: 105 | shape=(105, 1024) | supervised=80,163
Window document split matches manifest: PASS
Labels.txt order: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Checkpoint BIO map matches labels.txt: PASS
Manifest BIO order matches labels.txt: PASS
Strict model + BIO-head restore: PASS | LoRA modules: 48
Checkpoint epoch: 5
Stored validation F1: 0.10702425831617758
All validation tokens receive exactly one prediction: PASS

DECO